# Coastal water clarity

We tested ways to estimate suspended particulate matter (SPM, mg/L) and Secchi depth (m) from satellite reflectance. SPM measures suspended mass. Secchi depth is the depth at which a lowered disk disappears, with larger values indicating clearer water. Neither directly measures turbidity in NTU. Separate target lists let us use observations with just one of these measurements.

“Local” refers to the Belgian coast and nearby French and Dutch coastal/estuarine stations. We started with correlations and GLORIA field spectra for neural-network pretraining, then tested local transfer, simpler regressions, random forests, wind, a smaller S3 footprint and extra GLORIA satellite matchups. Neural training covered S2-SPM only. The conventional models covered both sensors and targets.

This notebook records the experiments in order, with their saved outputs. Set `PROJECT` to this folder and `GLORIA` to the original GLORIA CSVs. Some cells also read local satellite cutouts. Training stops if a named run folder already exists, to prevent overwriting results. You can read the outputs without rerunning the experiments. A new run needs separate output folders and matching path references.

For the final application, we chose to map SPM with both sensors. Secchi had fewer usable pairs, although some random-forest results were promising. Its exclusion reflects that narrower mapping scope. The experiments below show how we reached the choice.


## 1. Satellite data

We paired field measurements from [LifeWatch](https://doi.org/10.14284/652), [Rijkswaterstaat](https://rijkswaterstaatdata.nl/waterdata/) and [SOMLIT-Wimereux](https://doi.org/10.17882/96639) with satellite acquisitions within three hours. We used positive SPM samples from known near-surface depths of 0–5 m and separate Secchi readings that passed provider-specific quality checks.

Remote-sensing reflectance, $R_{rs}$ (sr⁻¹), is water-leaving radiance divided by downwelling irradiance just above the water. Atmospheric correction separates the water signal from atmospheric contributions. Each model input is an arithmetic mean over nearby valid pixels:

- Sentinel-2 (S2), MultiSpectral Instrument (MSI): Level-1C corrected with ACOLITE 20260421.0 using dark spectrum fitting and residual glint correction. A 5 × 5 window on the common 20 m grid covers 100 × 100 m and needs at least 20/25 common finite green/red/red-edge pixels with `l2_flags = 0`.
- Sentinel-3 (S3), Ocean and Land Colour Instrument (OLCI): Level-2 Water, with a 3 × 3 window of nominal 300 m pixels, about 900 × 900 m. All nine must pass the collection-specific water, cloud, glint and processing-quality mask. We divide Collection 003 directional water reflectance by π to obtain $R_{rs}$. Collection 004 is already BRDF-normalised $R_{rs}$, with viewing and illumination normalisation. We analyse the collections separately and use only 003 for modelling.

We select one image per sensor, collection and field sample from verified quality, coverage and time difference, without using the measured target. We inspect variation within windows but do not use it as an extra filter. Band means are not clipped. Columns beginning `raw__` contain these prepared values, not uncorrected Level-1 reflectance.

We use LifeWatch's mg/L unit field despite a conflicting parameter label. SOMLIT times are assumed to use Europe/Paris civil time, including summer time, without provider confirmation. Other timestamps use their recorded offsets. These unit and timing uncertainties can affect the results.

Sample IDs, acquisition dates and stations are recorded in the tables. Several stations can share an image, and one field sample can have both sensor pairs, so sample counts differ from counts of independent images. [Feature definitions](data/satellite_audit/feature_registry.csv).


In [ ]:
# 1: paths and imports
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()  # Folder containing this notebook.
DATA = PROJECT / 'data'
GLORIA = PROJECT / 'data' / 'GLORIA_2022'
assert DATA.is_dir(), 'Set PROJECT to the coastal_project folder.'


In [ ]:
# 1: accepted satellite-field observations
matchups = pd.read_csv(DATA / 'satellite_audit/analysis_matchups.csv',
                       dtype={'collection': str, 'station_id': str})
assert matchups['primary_verified'].all()
assert not matchups.duplicated(['sensor', 'collection', 'sample_id']).any()

matchups.groupby(['sensor', 'collection', 'target']).agg(
    samples=('sample_id', 'nunique'),
    acquisition_dates=('acquisition_date', 'nunique'),
    stations=('station_id', 'nunique')
)


### Correlations

We check whether stronger optical signals accompany higher SPM or different Secchi depths. Green/red/red-edge bands are S2 B3/B4/B5, about 560/665/704 nm, and S3 Oa06/Oa08/Oa11, 560/665/709 nm. NDTI, the Normalized Difference Turbidity Index, is `(red − green)/(red + green)`. We calculate it from the band means, with both above 10⁻⁶ sr⁻¹ to avoid unstable ratios.

Pearson $r$ measures a straight-line association. Spearman $\rho$ applies the calculation to ranks, showing whether values increase or decrease together without requiring a straight line. Both range from −1 to 1. We use unscaled inputs and untransformed targets in this table.

Each `n` counts rows with both a finite feature and target. We omit groups with fewer than ten observations or five acquisition dates. These pooled correlations include observations later reserved for testing. They describe associations, not prediction accuracy, and do not choose the final predictor.


In [ ]:
# 1: recalculate correlations from the saved observations
bands = {
    'S2': ['Rrs_B3', 'Rrs_B4', 'Rrs_B5', 'NDTI'],
    'S3': ['Rrs_Oa06', 'Rrs_Oa08', 'Rrs_Oa11', 'NDTI']
}
rows = []
for (sensor, collection, target), group in matchups.groupby(['sensor', 'collection', 'target']):
    for band in bands[sensor]:
        pairs = group[['raw__' + band, 'target_value', 'acquisition_date']].dropna()
        n_dates = pairs['acquisition_date'].nunique()
        if len(pairs) < 10 or n_dates < 5:
            continue
        x, y = pairs.iloc[:, 0], pairs['target_value']
        rows.append([sensor, collection, target, band, len(pairs), n_dates,
                     x.corr(y), x.rank().corr(y.rank())])

correlations = pd.DataFrame(rows, columns=[
    'sensor', 'collection', 'target', 'band', 'n', 'dates', 'Pearson r', 'Spearman rho'
]).set_index(['sensor', 'collection', 'target', 'band'])
correlations.round(3)


We compare green and red-edge reflectance and NDTI with the field measurements below. The plots use the saved training/development observations, so counts differ from the pooled table above. For S3, only Collection 003 is shown. Reserved test dates are not shown. Each point is one satellite–field pair from the sampling windows described above. Axes are linear, and counts are shown per panel. A missing input affects only its own panel. These plots describe associations without fitting a model.


In [ ]:
# A4: initial satellite–field comparisons at the reference stations
import numpy as np
import matplotlib.pyplot as plt

a4_roles = pd.read_csv(DATA / 'model_preparation/local_assignments.csv')
a4 = matchups.merge(a4_roles[['row_id', 'role']], on='row_id',
                    how='inner', validate='one_to_one')
a4 = a4.loc[a4['role'].eq('development')]
a4_bands = {
    'S2': [('raw__Rrs_B3', 'Green B3, Rrs (sr⁻¹)'),
           ('raw__Rrs_B5', 'Red edge B5, Rrs (sr⁻¹)'),
           ('raw__NDTI', 'NDTI (dimensionless)')],
    'S3': [('raw__Rrs_Oa06', 'Green Oa06, Rrs (sr⁻¹)'),
           ('raw__Rrs_Oa11', 'Red edge Oa11, Rrs (sr⁻¹)'),
           ('raw__NDTI', 'NDTI (dimensionless)')]
}
a4_out = PROJECT / 'figures/A4'
a4_out.mkdir(parents=True, exist_ok=True)

for sensor, features in a4_bands.items():
    fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharey='row', layout='constrained')
    for i, (target, label) in enumerate([('SPM', 'SPM (mg/L)'), ('Secchi', 'Secchi depth (m)')]):
        group = a4.loc[a4['sensor'].eq(sensor) & a4['target'].eq(target)]
        for ax, (feature, xlabel) in zip(axes[i], features):
            pairs = group[[feature, 'target_value', 'acquisition_date']].replace(
                [np.inf, -np.inf], np.nan).dropna()
            ax.scatter(pairs[feature], pairs['target_value'], s=26, alpha=.65,
                       color='#2878a5', edgecolors='none')
            ax.set_xlabel(xlabel)
            ax.set_title(f"n={len(pairs)}, {pairs['acquisition_date'].nunique()} dates",
                         loc='left', fontsize=10)
            ax.grid(alpha=.18)
            ax.set_axisbelow(True)
        axes[i, 0].set_ylabel('Field ' + label)
        axes[i, 0].set_ylim(bottom=0)
    fig.suptitle(f"Sentinel-{sensor[1]}: satellite and field measurements", fontsize=14)
    for suffix in ['png', 'svg']:
        fig.savefig(a4_out / f'{sensor}_initial_comparison.{suffix}', dpi=300)
    plt.show()


The plots suggest a broad tendency towards higher SPM and shallower Secchi depth at higher reflectance or NDTI. The S2 SPM points are widely scattered, especially at high concentrations. The sensors cover different observations and use different processing, so these plots cannot establish which sensor is more accurate.


## 2. GLORIA

[GLORIA](https://doi.pangaea.de/10.1594/PANGAEA.948492) contains field-measured reflectance spectra and water-quality observations from around the world ([Lehmann et al., 2023](https://www.nature.com/articles/s41597-023-01973-y)). A spectrum records reflectance across wavelengths. We use these labelled spectra to test whether more training data helps a model with few local observations.

We join metadata, labels, spectra and quality flags by `GLORIA_ID`, the observation identifier. TSS, total suspended solids, provides the source SPM label. Its g/m³ values equal mg/L numerically, although sampling and laboratory methods can differ. “Source” means GLORIA below, and “local” means our regional satellite–field pairs.

This first use of GLORIA does not need a satellite acquisition. Section 3 converts field spectra into approximate sensor-band values. In Section 11 we instead match GLORIA field samples to satellite images. Missing labels and reflectance are not filled.


In [ ]:
# 2: read the original GLORIA files
meta = pd.read_csv(GLORIA / 'GLORIA_meta_and_lab.csv').dropna(how='all')
qc = pd.read_csv(GLORIA / 'GLORIA_qc_flags.csv').dropna(how='all')
spectra = pd.read_csv(GLORIA / 'GLORIA_Rrs.csv').dropna(how='all')

meta = meta.set_index('GLORIA_ID', verify_integrity=True)
qc = qc.set_index('GLORIA_ID', verify_integrity=True)
spectra = spectra.set_index('GLORIA_ID', verify_integrity=True)
assert set(meta.index) == set(qc.index) == set(spectra.index)

gloria = meta.join(qc, validate='one_to_one').join(spectra, validate='one_to_one')
gloria['waterbody_key'] = gloria['Country'].str.strip() + ' / ' + gloria['Site_name'].str.strip()
pd.Series({'joined observations': len(gloria), 'wavelengths': len(spectra.columns),
           'Flagged = 0': gloria['Flagged'].eq(0).sum(),
           'Flagged = 1': gloria['Flagged'].eq(1).sum()}, name='n')


### Quality selection

We use GLORIA's estuary and coastal-ocean categories, `Water_body_type` 2 and 3, with separate lists for positive TSS and Secchi labels. Selection concerns reflectance quality, without imposing a turbidity threshold.

A flag of 1 means an issue was detected, 0 means it was not detected, and missing means it was not assessed. We inspect the individual flags instead of excluding every row with `Flagged = 1`.

| Flag | What it checks | Use for green/red/red-edge inputs |
| --- | --- | --- |
| Noisy_blue | Noise at 350–400 nm | Warning only |
| Negative_uv_slope | Unusual slope at 350–420 nm | Warning only |
| Noisy_red | Noise at 750–900 nm, not around 705 nm | Warning for core bands, require 0 for optional NIR |
| Oxygen_signal | Artefact near 762 nm | Warning only |
| Baseline_shift | Possible overall reflectance offset | Exclude |
| QWIP_fail | Visible spectral shape outside the quality-check envelope | Exclude |
| Suspect | Expert judgement of suspicious reflectance | Exclude |
| Flagged | Combined summary of checks | Comparison only |

The core bands respond over roughly 522–736 nm, depending on platform. Warnings outside this range alone do not exclude a spectrum. All three exclusion flags must equal zero, so a missing flag does not pass. A spectrum that passes can still contain errors.

QWIP means Quality Water Index Polynomial. It compares spectral shape with an expected envelope and can flag unusual real spectra as well as errors. We chose to exclude these flags conservatively. [Flag implementation](https://github.com/dmaciel123/gloria_flags), [QWIP paper](https://doi.org/10.3389/frsen.2022.869611). The count table shows losses at each step. Flags overlap, so their separate counts cannot be added.


In [ ]:
# 2: inspect each flag before choosing the exclusion list
coastal = gloria.loc[gloria['Water_body_type'].isin([2, 3])].copy()
flag_counts = pd.DataFrame({
    'SPM flagged': coastal.loc[coastal['TSS'].gt(0), qc.columns].eq(1).sum(),
    'Secchi flagged': coastal.loc[coastal['Secchi_depth'].gt(0), qc.columns].eq(1).sum(),
    'SPM unknown': coastal.loc[coastal['TSS'].gt(0), qc.columns].isna().sum(),
    'Secchi unknown': coastal.loc[coastal['Secchi_depth'].gt(0), qc.columns].isna().sum()
})
flag_counts


In [ ]:
# 2: apply the explicit working rule, keeping the original data
reject_flags = ['Baseline_shift', 'Suspect', 'QWIP_fail']
quality_ok = pd.Series(True, index=coastal.index)
labels = ['TSS', 'Secchi_depth']
selection = {'Coastal + positive label': [coastal[col].gt(0).sum() for col in labels]}

for flag in reject_flags:
    quality_ok &= coastal[flag].eq(0)
    selection['After ' + flag] = [
        (quality_ok & coastal[col].gt(0)).sum() for col in labels
    ]

gloria_spm = coastal.loc[quality_ok & coastal['TSS'].gt(0)].copy()
gloria_secchi = coastal.loc[quality_ok & coastal['Secchi_depth'].gt(0)].copy()
selection['Comparison: old Flagged = 0'] = [
    (coastal['Flagged'].eq(0) & coastal[col].gt(0)).sum() for col in labels
]
pd.DataFrame(selection, index=['SPM', 'Secchi']).T


In [ ]:
# 2: selected candidates by reported water-body type
waterbodies = {2: 'Estuary', 3: 'Coastal ocean'}
by_waterbody = pd.DataFrame({
    'SPM': gloria_spm.groupby('Water_body_type').size(),
    'Secchi': gloria_secchi.groupby('Water_body_type').size()
}).rename(index=waterbodies)
by_waterbody.index.name = 'water body'
by_waterbody


### Wavelength coverage

The table counts available individual wavelengths in the quality-selected coastal/estuarine spectra. Complete satellite bands need coverage across a wavelength range, which we check next. The 865 nm column checks optional near-infrared (NIR) coverage. It is not a model input. We do not fill gaps, clip reflectance or add noise.


In [ ]:
# 2: inspect available reflectance at selected wavelengths
coverage = pd.DataFrame({
    'TSS / SPM': gloria_spm[spectra.columns].notna().sum(),
    'Secchi': gloria_secchi[spectra.columns].notna().sum()
})
coverage.loc[[f'Rrs_{w}' for w in [443, 490, 560, 665, 705, 709, 865]]]


## 3. Sensor-band conversion

A satellite band covers a wavelength range and weights some wavelengths more strongly than others. We average each GLORIA spectrum with the sensor's spectral response instead of reading its centre wavelength alone. The core bands are green/red/red edge: S2 MSI B3/B4/B5 and S3 OLCI Oa06/Oa08/Oa11.

$$R_{rs,b} \approx \frac{\sum_\lambda R_{rs}(\lambda)\,S_b(\lambda)}{\sum_\lambda S_b(\lambda)}$$

$R_{rs}(\lambda)$ is the measured spectrum and $S_b(\lambda)$ is band $b$'s response at wavelength $\lambda$. Dividing by the sum of responses gives a weighted average in sr⁻¹. This follows the sensor-band conversion in [Pahlevan et al. (2022), §3.1](https://doi.org/10.1016/j.rse.2021.112860), [ACOLITE's open code](https://github.com/acolite/acolite/blob/main/acolite/shared/rsr_convolute_dict.py) and a [GLORIA neural-network study (2024), §2.1](https://doi.org/10.3389/frsen.2024.1383147). This is an approximation because radiance and irradiance are not integrated separately ([Burggraaff, 2020](https://doi.org/10.1364/OE.391470)).

We use separate responses for S2A/B/C and S3A/B. MSI tables match the saved processing files. OLCI mean responses are shifted to Level-2 nominal wavelengths. [Versions and source files](data/srf_manifest.json).

We interpolate the response curves onto GLORIA's 1 nm grid, but do not interpolate reflectance. If a wavelength with positive response is missing, the simulated band is missing too. We do not average over a partial band. Differences in field/satellite viewing geometry and atmospheric correction are not resolved by this conversion.


In [ ]:
# 3: align responses and average each spectrum over the full band
import numpy as np

rsr_data = pd.read_csv(DATA / 'spectral_response_samples.csv')
wavelengths = spectra.columns.str.removeprefix('Rrs_').astype(int).to_numpy()
simulated, responses = {}, {}

for (platform, band), curve in rsr_data.groupby(['platform', 'band'], sort=False):
    shifted = curve['wavelength_nm'] + curve['shift_nm']
    assert shifted.between(wavelengths.min(), wavelengths.max()).all()
    # Interpolate the response only. Missing reflectance stays missing.
    response = np.interp(wavelengths, shifted, curve['response'], left=0, right=0)
    responses[platform + '_' + band] = response
    use = response > 0
    weights = response[use] / response[use].sum()
    if platform not in simulated:
        simulated[platform] = pd.DataFrame(index=spectra.index)
    simulated[platform][band] = spectra.loc[:, use].to_numpy() @ weights

responses = pd.DataFrame(responses, index=wavelengths)
band_counts = []
for platform, frame in simulated.items():
    band_counts.append([platform,
                        frame.reindex(gloria_spm.index).dropna().shape[0],
                        frame.reindex(gloria_secchi.index).dropna().shape[0]])
pd.DataFrame(band_counts, columns=['platform', 'SPM: all 3 bands', 'Secchi: all 3 bands'])


### Band indices

We store converted bands in `simulated` and add five derived inputs to copies called `simulated_with_indices`. Let $G$, $R$ and $E$ be green, red and red-edge reflectance:

| Input name | Formula |
| --- | --- |
| NDTI | $(R-G)/(R+G)$ |
| red_green | $R/G$ |
| red_edge_red | $E/R$ |
| red_edge_green | $E/G$ |
| ND_rededge_red | $(E-R)/(E+R)$ |

The indices are dimensionless combinations of the same three bands, not additional measurements. NDTI gives the same ordering as positive red/green. Each required band must exceed 10⁻⁶ sr⁻¹, as in the regional index rule. If one fails, that index is missing. This does not change the band values or observation lists.


In [ ]:
# 3: keep the band-only tables and make copies with indices
simulated_with_indices = {platform: frame.copy() for platform, frame in simulated.items()}

for platform, frame in simulated_with_indices.items():
    core = ['B3', 'B4', 'B5'] if platform.startswith('S2') else ['Oa06', 'Oa08', 'Oa11']
    green, red, red_edge = (frame[col].where(frame[col].gt(1e-6)) for col in core)
    frame['NDTI'] = (red - green) / (red + green)
    frame['red_green'] = red / green
    frame['red_edge_red'] = red_edge / red
    frame['red_edge_green'] = red_edge / green
    frame['ND_rededge_red'] = (red_edge - red) / (red_edge + red)

simulated_with_indices['S2A'].head().round(5)


### Earlier optional retrievals

We also calculated NIR and two published-formula estimates for exploration. These are excluded from the models below. NIR uses S2 B8A or S3 Oa17, around 865 nm, with full spectral coverage and `Noisy_red = 0`. Missing or unassessed NIR quality affects these calculations only.

Nechad estimates SPM, while Dogliotti estimates turbidity in FNU. Neither replaces a field label. Both use water reflectance $\rho=\pi R_{rs}$ instead of $R_{rs}$ directly.

The [Nechad et al. (2010)](https://doi.org/10.1016/j.rse.2009.11.022) form used here is `355.85 × ρred / (1 − ρred/0.1728)`, with `0 ≤ ρred < 0.5 × 0.1728`. These are the project's fixed 665 nm coefficients and validity limit, without local calibration. High-red observations outside this range have no Nechad estimate. Requiring one would remove those water conditions from training, so we excluded it from the inputs.

[Dogliotti et al. (2015), §2.1 and Table 2](https://doi.org/10.1016/j.rse.2014.09.020) switches from red to NIR as reflectance increases, blending over a red-reflectance interval of 0.05–0.07. We apply its 645/859 nm coefficients to the available approximately 665/865 nm bands. This uses nearby bands without recalibrating the coefficients. Inputs must be nonnegative with valid denominators. The code evaluates the relevant branch only.

The table counts available estimates, without testing their accuracy. “Nechad: Secchi” counts Secchi-labelled records with an available Nechad SPM estimate, not predicted Secchi depth. Missing estimates do not exclude core observations. [Formula registry](data/satellite_audit/feature_registry.csv).


In [ ]:
# 3: optional fixed retrievals, kept separate from the core features
nir_rsr = pd.read_csv(DATA / 'nir_response_samples.csv')
retrievals, retrieval_counts = {}, []

for platform, frame in simulated.items():
    curve = nir_rsr.loc[nir_rsr['platform'].eq(platform)]
    shifted = curve['wavelength_nm'] + curve['shift_nm']
    assert shifted.between(wavelengths.min(), wavelengths.max()).all()
    response = np.interp(wavelengths, shifted, curve['response'], left=0, right=0)
    use = response > 0
    nir = pd.Series(spectra.loc[:, use].to_numpy() @ (response[use] / response[use].sum()),
                    index=spectra.index)
    nir = nir.where(qc['Noisy_red'].eq(0))
    red_band = 'B4' if platform.startswith('S2') else 'Oa08'
    rho_red, rho_nir = np.pi * frame[red_band], np.pi * nir

    # Nechad: same 665 nm coefficients and 0.5 C limit as the satellite audit.
    rho = rho_red.where(rho_red.ge(0) & rho_red.lt(0.5 * 0.1728))
    nechad = 355.85 * rho / (1 - rho / 0.1728)

    # Dogliotti: require both inputs, then evaluate only active branches.
    valid = (np.isfinite(rho_red) & np.isfinite(rho_nir)
             & rho_red.ge(0) & rho_nir.ge(0))
    red, nir_rho = rho_red.where(valid), rho_nir.where(valid)
    weight = ((red - 0.05) / 0.02).clip(0, 1)
    r = red.where(weight.lt(1) & red.lt(0.1641))
    n = nir_rho.where(weight.gt(0) & nir_rho.lt(0.2112)
                      & (1 - nir_rho / 0.2112).abs().gt(1e-8))
    t_red = 228.1 * r / (1 - r / 0.1641)
    t_nir = 3078.9 * n / (1 - n / 0.2112)
    dogliotti = pd.Series(np.nan, index=frame.index)
    dogliotti.loc[weight.eq(0)] = t_red
    dogliotti.loc[weight.eq(1)] = t_nir
    blend = weight.gt(0) & weight.lt(1)
    dogliotti.loc[blend] = ((1 - weight) * t_red + weight * t_nir).loc[blend]

    retrievals[platform] = pd.DataFrame({
        curve['band'].iloc[0]: nir,
        'Nechad2010_ACOLITE_form': nechad,
        'Dogliotti2015_nearest_available': dogliotti
    })
    retrieval_counts.append([platform,
        nechad.reindex(gloria_spm.index).notna().sum(),
        nechad.reindex(gloria_secchi.index).notna().sum(),
        dogliotti.reindex(gloria_spm.index).notna().sum(),
        dogliotti.reindex(gloria_secchi.index).notna().sum()])

pd.DataFrame(retrieval_counts, columns=[
    'platform', 'Nechad: SPM', 'Nechad: Secchi', 'Dogliotti: SPM', 'Dogliotti: Secchi'
])


### Model input candidates

The candidate list contains three bands and five indices, eight inputs in total. It excludes NIR, Nechad and Dogliotti. The quantity to predict is still measured SPM or Secchi depth. This cell copies the candidate tables. Section 6 checks input availability before model training.


In [ ]:
# 3: non-NIR input candidates, with all original rows retained
candidate_inputs = {
    platform: frame.copy()
    for platform, frame in simulated_with_indices.items()
}
pd.DataFrame({'S2': candidate_inputs['S2A'].columns,
              'S3': candidate_inputs['S3A'].columns})


## 4. Inspect spectra

The six spectra show examples with no detected warnings, accepted warnings and rejected flags. We chose them from flags and reflectance, without using SPM or Secchi values. We inspect their shapes, negative values and the wavelengths used by the bands.

Grey areas show S2A band-response support and orange dots show the response-weighted band values. Each panel has its own vertical scale, so heights are not directly comparable between panels. Change the observation IDs to inspect other spectra. These plots do not alter selection.


In [ ]:
# 4: manual inspection, two examples per column
import matplotlib.pyplot as plt

example_ids = ['GID_4917', 'GID_3430', 'GID_5413',
               'GID_5560', 'GID_1724', 'GID_730']
wavelengths = spectra.columns.str.removeprefix('Rrs_').astype(int)
s2a = responses.filter(like='S2A_')
centres = s2a.mul(s2a.index, axis=0).sum() / s2a.sum()
fig, axes = plt.subplots(2, 3, figsize=(11, 5.5), sharex=True)

for ax, gid in zip(axes.flat, example_ids):
    ax.plot(wavelengths, spectra.loc[gid], color='0.2', linewidth=1)
    for name in s2a:
        support = s2a.index[s2a[name].gt(0)]
        ax.axvspan(support.min(), support.max(), color='0.92', zorder=0)
    ax.scatter(centres, simulated['S2A'].loc[gid], color='#d76a19', s=20, zorder=3)
    issues = qc.loc[gid].drop('Flagged')
    warnings = ', '.join(issues.index[issues.eq(1)]) or 'no detected warnings'
    status = 'QC kept' if quality_ok.loc[gid] else 'QC rejected'
    ax.set_title(f'{gid}: {status}\n{warnings}', fontsize=9)
    ax.axhline(0, color='0.6', linewidth=0.5)
    ax.set_xlim(350, 900)

fig.supxlabel('Wavelength (nm)')
fig.supylabel('Rrs (sr⁻¹)')
fig.tight_layout()
plt.show()


## 5. Pretraining plan

We first train on labelled GLORIA spectra, then update the network with local satellite–field pairs. This pretraining and fine-tuning sequence tests whether a relationship learned from more observations gives a useful starting point for the small local dataset.

[Zhao (2026)](https://doi.org/10.1016/j.jag.2026.105294) and [Gaur (2025)](https://doi.org/10.1007/s11783-025-1930-x) tested transfer learning for water quality. [Saranathan (2024)](https://doi.org/10.3389/frsen.2024.1383147) used GLORIA sensor-band training, IQR scaling and log-transformed TSS. We use a small multilayer perceptron (MLP), a fully connected neural network. The cited studies guide the approach, but we do not reproduce their architectures or complete workflows.

- We prepare four datasets: S2-SPM, S2-Secchi, S3-SPM and S3-Secchi. Neural training covers S2-SPM only. Separate target lists allow samples with one label. S3 modelling uses Collection 003.
- The network has eight inputs, two “hidden” layers of 15 then 15 or 32 then 16 units, and one output. ReLU activations replace negative intermediate values with zero, allowing a nonlinear relationship. [Niroumand-Jadidi & Bovolo (2022), Methods and Fig. 1](https://doi.org/10.5194/isprs-annals-V-3-2022-339-2022) informed the 15–15 candidate, although its model differs from ours.
- We split GLORIA water-body/coastal-area groups approximately 70/15/15% into training/validation/test, seed 42. Training learns weights, validation chooses settings and stopping time, and test groups are set aside. Platform versions of an observation stay together and share its total weight.
- Scale inputs as `(value − training median) / training IQR`, with IQR the 75th minus 25th percentile. These values are calculated from GLORIA training data and used for local transfer and the random-start control.
- Compare mean squared error (MSE) on log10 or square-root targets. MSE averages squared differences. Both transforms compress high concentrations, the square root less strongly. We compare candidates using validation log10-RMSE after converting predictions to mg/L, since the training losses have different scales. [Transform definitions](https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.boxcox.html).
- Use Adam, learning rate 0.001, weight decay 0.0001, batches of 32 and up to 300 epochs. An epoch is one pass through the training set. Stop after 30 epochs without validation improvement and reload the best weights.
- Set aside about 20% of local acquisition dates. Divide the others into three groups to compare updating the output layer alone with updating the whole network at a smaller learning rate. Include random-start, random-forest and constant controls.
- A feature must be available in at least 75% of GLORIA training and local development observations, counted before dropping incomplete rows. We do not fill missing labels or inputs, or replace negative reflectance with zero.

MAE is the average absolute error. RMSE gives larger errors more weight. Both use mg/L for SPM or metres for Secchi. R² is 1 for perfect predictions, 0 for predicting the evaluation-set mean and negative for worse errors. Log10-RMSE measures disagreement on the log scale and has no units. Lower MAE, RMSE and log10-RMSE are better. We use log10-RMSE for selection to emphasise proportional errors. Bias is mean prediction minus measurement, so negative values indicate underestimation. [Metric definitions](https://scikit-learn.org/1.7/modules/model_evaluation.html#regression-metrics).

We chose the fractions, thresholds, network sizes and optimiser settings for this comparison. [Structured-data validation guidance](https://doi.org/10.1111/ecog.02881) explains why related observations should be grouped. We did not run the proposed noise augmentation or lake/river expansion. Sections 8–10 show the S2-SPM experiment.


## 6. Splits and usable inputs

GLORIA validation asks whether the model predicts spectra from other coastal areas. Local validation asks whether it predicts other dates within the sampled region. Local stations can appear in both training and evaluation.

We group GLORIA records by reported site, then merge sites with observations within 25 km, including chains of links. The listed Chesapeake Bay tributaries are also grouped with the bay. This gives 35 geographic groups. We chose the distance to group nearby sites, not to reject observations. It does not guarantee independent groups.

We shuffle the sorted GLORIA groups with seed 42. Five go to test, five to validation and 25 to training, approximately 15/15/70% of groups. Sample shares differ because group sizes differ. Targets and platform versions share the assignments. Group names identify a representative site without changing the reported water-body names.

We separately shuffle local acquisition dates with seed 42 and reserve 20%, rounded up, as `test`. The rest are labelled `development` for training and model selection. We divide these dates into three groups, called `fold` in the code. Train on two and predict the third, repeating until each observation has a prediction from a model that excluded its date. When choosing SVR settings or regression formulas, we split the training part again. These are the `inner` folds, and the initially excluded group is the `outer` check. [Scikit-learn grouped validation](https://scikit-learn.org/1.7/modules/cross_validation.html#group-k-fold).

All comparisons use the saved assignments. Earlier correlations included local test observations, and Section 10 shows S2 test performance before the later experiments. These test rows are excluded from fitting, but that history limits the independence of later conclusions. The splits test other dates within this region, not forecasting or new coastlines.


In [ ]:
# 6: group nearby reported sites without removing observations
PREP = DATA / 'model_preparation'
PREP.mkdir(exist_ok=True)

def save_fixed(frame, filename):
    path = PREP / filename
    contents = frame.to_csv(index=False)
    if path.exists():
        assert path.read_text() == contents, f'{filename} changed. Review before replacing.'
    else:
        frame.to_csv(path, index=False)

eligible_ids = gloria_spm.index.union(gloria_secchi.index).sort_values()
geography = gloria.loc[eligible_ids, ['waterbody_key', 'Latitude', 'Longitude']].copy()
codes, site_names = pd.factorize(geography['waterbody_key'], sort=True)
lat, lon = np.deg2rad(geography[['Latitude', 'Longitude']].to_numpy()).T
h = (np.sin((lat[:, None] - lat)/2)**2
     + np.cos(lat[:, None])*np.cos(lat)*np.sin((lon[:, None] - lon)/2)**2)
distance = 12742*np.arcsin(np.sqrt(np.clip(h, 0, 1)))
i, j = np.where(np.triu(distance <= 25, 1) & (codes[:, None] != codes))
links = pd.DataFrame({'site_a': site_names[codes[i]], 'site_b': site_names[codes[j]],
                      'distance_km': distance[i, j]})
links = links.groupby(['site_a', 'site_b'], as_index=False)['distance_km'].min()

# Named tributaries are kept with their parent bay even beyond 25 km.
chesapeake = ['Chesapeake Bay, MD', 'Baltimore Harbor, MD', 'Monie Bay, MD',
              'Otter Point Creek, MD', 'St. Leonard Creek, MD']
bay_sites = geography.loc[gloria.loc[eligible_ids, 'Site_name'].isin(chesapeake),
                         'waterbody_key'].unique()
extra = pd.DataFrame({'site_a': bay_sites[0], 'site_b': bay_sites[1:],
                      'distance_km': np.nan})
edges = pd.concat([links.assign(reason='within 25 km'),
                   extra.assign(reason='named Chesapeake tributary')], ignore_index=True)
group = {site: site for site in site_names}
for a, b in edges[['site_a', 'site_b']].itertuples(index=False, name=None):
    old, new = group[b], group[a]
    group = {site: new if value == old else value for site, value in group.items()}
geography['geographic_group'] = geography['waterbody_key'].map(group)
save_fixed(edges, 'geographic_links.csv')
save_fixed(geography.reset_index(), 'geographic_groups.csv')
pd.Series({'observations': len(geography), 'reported sites': len(site_names),
           'geographic groups': geography['geographic_group'].nunique(),
           'missing coordinates, retained in reported site': geography['Latitude'].isna().sum()})


In [ ]:
# 6: freeze GLORIA groups before checking feature availability
groups = np.sort(geography['geographic_group'].unique())
order = np.random.default_rng(42).permutation(groups)
n_test = round(0.15*len(order))
n_val = round(0.15*len(order))
roles = dict.fromkeys(order, 'train')
roles.update(dict.fromkeys(order[:n_test], 'test'))
roles.update(dict.fromkeys(order[n_test:n_test+n_val], 'validation'))
source_roles = geography[['waterbody_key', 'geographic_group']].copy()
source_roles['role'] = source_roles['geographic_group'].map(roles)
source_roles['SPM_eligible'] = source_roles.index.isin(gloria_spm.index)
source_roles['Secchi_eligible'] = source_roles.index.isin(gloria_secchi.index)
save_fixed(source_roles.reset_index(), 'gloria_assignments.csv')
assert source_roles.groupby('geographic_group')['role'].nunique().eq(1).all()
source_roles.groupby('role').agg(groups=('geographic_group', 'nunique'),
    SPM=('SPM_eligible', 'sum'), Secchi=('Secchi_eligible', 'sum')).reindex(
    ['train', 'validation', 'test'])


In [ ]:
# 6: reserve whole local dates across sensors and targets
local = matchups.loc[matchups['sensor'].eq('S2') | matchups['collection'].eq('003')].copy()
assert local.groupby('sample_id')['acquisition_date'].nunique().eq(1).all()
assert local.groupby('product_id')['acquisition_date'].nunique().eq(1).all()
dates = np.sort(local['acquisition_date'].unique())
test_dates = np.random.default_rng(42).permutation(dates)[:int(np.ceil(0.2*len(dates)))]
local['role'] = np.where(local['acquisition_date'].isin(test_dates), 'test', 'development')
local['platform'] = local['product_name'].str[:3]
local_roles = local[['row_id', 'sample_id', 'sensor', 'collection', 'target', 'platform',
                     'acquisition_date', 'product_id', 'role']].sort_values('row_id')
save_fixed(local_roles, 'local_assignments.csv')
assert local.groupby('acquisition_date')['role'].nunique().eq(1).all()
local.groupby(['sensor', 'target', 'role']).agg(
    observations=('sample_id', 'nunique'), dates=('acquisition_date', 'nunique'))


We count the availability of each candidate before removing rows with missing features. The threshold is 75% in both GLORIA training and local development data, separately by sensor and target. Every platform variant must pass for GLORIA. Neither GLORIA validation/test groups nor reserved local test dates choose this list. We use input availability across all local development rows, including later validation dates, without using their target values. This check is done once, not inside each later training fold.

Passing 75% does not give every row a complete input set. The next table counts rows with all selected features. Models use those complete rows without filling gaps. The source tables also include incomplete records. Platform versions do not count as extra independent observations.

This check excludes NIR, Dogliotti and Nechad. High red reflectance alone does not reject a sample. Spectral-quality checks and ratio safeguards still apply. These cells do not scale inputs or train a model.


In [ ]:
# 6: apply the agreed gate without using validation or test rows
index_columns = {'NDTI': 'NDTI', 'red_green': 'red_green_ratio',
    'red_edge_red': 'rededge_red_ratio', 'red_edge_green': 'rededge_green_ratio',
    'ND_rededge_red': 'ND_rededge_red'}
local_inputs, selected_inputs, availability_rows = {}, {}, []

for sensor in ['S2', 'S3']:
    platforms = [p for p in candidate_inputs if p.startswith(sensor)]
    columns = candidate_inputs[platforms[0]].columns.tolist()
    raw_columns = ['raw__' + index_columns.get(c, 'Rrs_' + c) for c in columns]
    for target, pool in [('SPM', gloria_spm), ('Secchi', gloria_secchi)]:
        source_ids = pool.index.intersection(source_roles.index[source_roles['role'].eq('train')])
        part = local.loc[local['sensor'].eq(sensor) & local['target'].eq(target)]
        values = part[raw_columns].copy().replace([np.inf, -np.inf], np.nan)
        values.columns = columns
        local_inputs[sensor, target] = values
        source_counts = pd.DataFrame({p: np.isfinite(candidate_inputs[p].loc[source_ids]).sum()
                                      for p in platforms})
        dev = values.loc[part.index[part['role'].eq('development')]]
        source_fraction = source_counts.min(axis=1)/len(source_ids)
        local_fraction = dev.notna().mean()
        keep = source_fraction.ge(0.75) & local_fraction.ge(0.75)
        selected_inputs[sensor, target] = keep.index[keep].tolist()
        for feature in columns:
            availability_rows.append([sensor, target, feature, len(source_ids),
                int(source_counts.loc[feature].min()), len(dev), int(dev[feature].notna().sum()),
                source_fraction[feature], local_fraction[feature], bool(keep[feature])])

availability = pd.DataFrame(availability_rows, columns=['sensor', 'target', 'feature',
    'GLORIA_train_n', 'GLORIA_min_platform_available_n', 'local_dev_n', 'local_available_n',
    'GLORIA_fraction', 'local_fraction', 'keep'])
save_fixed(availability, 'feature_availability_no_nechad.csv')
availability.groupby(['sensor', 'target']).agg(
    candidates=('feature', 'size'), retained=('keep', 'sum'),
    GLORIA_min_fraction=('GLORIA_fraction', 'min'),
    local_min_fraction=('local_fraction', 'min')).round(3)


In [ ]:
# 6: show joint completeness, without selecting or imputing rows
count_rows = []
for (sensor, target), features in selected_inputs.items():
    assert features, f'No features passed for {sensor}, {target}'
    pool = gloria_spm if target == 'SPM' else gloria_secchi
    for platform, frame in candidate_inputs.items():
        if not platform.startswith(sensor):
            continue
        for role in ['train', 'validation', 'test']:
            ids = pool.index.intersection(source_roles.index[source_roles['role'].eq(role)])
            complete = np.isfinite(frame.loc[ids, features]).all(axis=1)
            count_rows.append(['GLORIA', platform, target, role, len(ids), int(complete.sum())])
    part = local.loc[local['sensor'].eq(sensor) & local['target'].eq(target)]
    for role in ['development', 'test']:
        ids = part.index[part['role'].eq(role)]
        complete = local_inputs[sensor, target].loc[ids, features].notna().all(axis=1)
        count_rows.append(['local', sensor, target, role, len(ids), int(complete.sum())])

complete_counts = pd.DataFrame(count_rows, columns=[
    'dataset', 'sensor_platform', 'target', 'role', 'eligible', 'all_inputs_present'])
complete_counts['incomplete'] = complete_counts['eligible'] - complete_counts['all_inputs_present']
save_fixed(complete_counts, 'complete_input_counts_no_nechad.csv')
complete_counts.assign(sensor=complete_counts['sensor_platform'].str[:2]).groupby(
    ['dataset', 'sensor', 'target', 'role']).agg(
    eligible=('eligible', 'first'), all_inputs_present=('all_inputs_present', 'min'),
    incomplete=('incomplete', 'max'))


All eight candidates pass. Some spectra lack full band coverage and some local ratios fail the positive-input safeguard. The table shows how many rows have every input. Excluding Nechad from the input list avoids losing high-red observations because of its validity limit.

Secchi validation covers only three GLORIA geographic groups. The local S2 Secchi test has eight observations from four dates. Such small sets give little confidence in a model ranking. We did not train a Secchi neural network.


## 7. Optional S2-SPM benchmark: MDN

We first check a published model. A mixture density network (MDN) represents several possible output distributions. We use point predictions from the published [MDN-STREAM MSI weights](https://github.com/STREAM-RS/MDN-STREAM/tree/79d1b47a992d25b134bdfe4d10fe7f4e7e30e6d1/Weights/MSI), without local fitting or uncertainty estimates.

The model uses S2 $R_{rs}$ bands B1–B7 and its saved input/output scaling. The local `mdn_msi.py` reader reconstructs the published calculation and takes the median across ten model rounds. It requires h5py. Its seven bands differ from our three bands plus five indices, so this comparison changes the inputs as well as the architecture.

We evaluate local S2-SPM development data, without test predictions, clipping or fine-tuning. Overlap with the published model's training samples has not been checked. This limits how independently we can interpret the score.


In [ ]:
# 7: frozen MDN predictions on S2-SPM development observations only
%pip install h5py==3.14.0
%pip install mdn_msi --quiet
from mdn_msi import predict_tss

mdn_bands = ['raw__Rrs_B' + str(i) for i in range(1, 8)]
mdn_dev = local.loc[local['sensor'].eq('S2') & local['target'].eq('SPM')
                    & local['role'].eq('development')].copy()
assert mdn_dev[[b + '__n' for b in mdn_bands]].ge(20).all().all()
mdn_dev['MDN_TSS_mg_l'] = predict_tss(mdn_dev[mdn_bands].to_numpy(), DATA / 'mdn_msi')
observed, predicted = mdn_dev['target_value'], mdn_dev['MDN_TSS_mg_l']
error = predicted - observed
pd.Series({'observations': len(mdn_dev), 'dates': mdn_dev['acquisition_date'].nunique(),
    'MAE (mg/L)': error.abs().mean(), 'RMSE (mg/L)': np.sqrt((error**2).mean()),
    'R2': 1 - (error**2).sum()/((observed-observed.mean())**2).sum(),
    'log10 RMSE': np.sqrt(((np.log10(predicted)-np.log10(observed))**2).mean())
}, name='Frozen MDN, development').round(3)


The published MDN gives development R² = −0.141, worse than predicting the mean of these measurements. It does not capture their SPM variation well. We did not fit or update the MDN locally, so this result describes its use as published on these observations.


## 8. First run: S2-SPM pretraining

We train the MLPs on GLORIA spectra converted to S2 bands and indices, using the source training and validation groups. The reserved GLORIA geographic test groups are not evaluated here.

S2A/B/C responses produce several versions of each spectrum. An observation has total weight one, split equally across its available complete-input versions. Weighted training medians and IQRs define scaling. Validation uses those training values.

Four candidates combine 15–15 or 32–16 hidden layers with log10 or square-root targets. A “checkpoint” stores the weights at an epoch. We reload the checkpoint with the lowest valid validation log10-RMSE. Negative square-root outputs are invalid, and we do not square them into positive concentrations. If any validation prediction is invalid, that epoch receives an infinite selection score.

The run uses PyTorch 2.9.1, NumPy ≥2 and Matplotlib on CPU. Models, learning curves and predictions are saved in `RUN`. The cell stops if the folder exists. This section makes no local or test predictions.


In [ ]:
# 8: assemble training and validation only
import torch

features = selected_inputs['S2', 'SPM']
source_ids = gloria_spm.index.intersection(
    source_roles.index[source_roles['role'].isin(['train', 'validation'])])
source = pd.concat([
    frame.loc[source_ids, features].assign(GLORIA_ID=source_ids, platform=platform)
    for platform, frame in candidate_inputs.items() if platform.startswith('S2')
], ignore_index=True)
source = source.join(source_roles[['role', 'geographic_group']], on='GLORIA_ID')
source = source.join(gloria_spm[['TSS']], on='GLORIA_ID')
source = source.loc[np.isfinite(source[features]).all(axis=1)].copy()
source['weight'] = 1/source.groupby('GLORIA_ID')['platform'].transform('size')
train = source.loc[source['role'].eq('train')].reset_index(drop=True)
valid = source.loc[source['role'].eq('validation')].reset_index(drop=True)
assert not set(train['GLORIA_ID']) & set(valid['GLORIA_ID'])
assert not set(train['geographic_group']) & set(valid['geographic_group'])
assert np.allclose(source.groupby('GLORIA_ID')['weight'].sum(), 1)
source.groupby('role').agg(observations=('GLORIA_ID', 'nunique'),
    platform_versions=('GLORIA_ID', 'size'), groups=('geographic_group', 'nunique'))


In [ ]:
# 8: weighted quantiles give each original observation equal influence
q25, centre, q75 = np.quantile(train[features].to_numpy(), [0.25, 0.5, 0.75],
    axis=0, weights=train['weight'].to_numpy(), method='inverted_cdf')
spread = np.where(q75 > q25, q75-q25, 1)
x_train = torch.tensor((train[features].to_numpy()-centre)/spread, dtype=torch.float32)
x_valid = torch.tensor((valid[features].to_numpy()-centre)/spread, dtype=torch.float32)
y_train = torch.tensor(train['TSS'].to_numpy(), dtype=torch.float32)
w_train = torch.tensor((train['weight']/train['weight'].mean()).to_numpy(), dtype=torch.float32)
y_valid = valid['TSS'].to_numpy()
w_valid = valid['weight'].to_numpy()
pd.DataFrame({'median': centre, 'IQR': spread}, index=features).round(5)


In [ ]:
# 8: same optimiser and batch order for each candidate
import copy

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)

def invert_target(z, transform):
    with np.errstate(over='ignore', invalid='ignore'):
        return 10.0**z if transform == 'log10' else np.where(z >= 0, z*z, np.nan)

def pretrain(hidden, transform):
    torch.manual_seed(42)
    model = torch.nn.Sequential(torch.nn.Linear(len(features), hidden[0]), torch.nn.ReLU(),
        torch.nn.Linear(hidden[0], hidden[1]), torch.nn.ReLU(), torch.nn.Linear(hidden[1], 1))
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
    target = torch.log10(y_train) if transform == 'log10' else torch.sqrt(y_train)
    generator = torch.Generator().manual_seed(42)
    best, best_state, best_epoch, stale = np.inf, None, None, 0
    history = []
    for epoch in range(1, 301):
        model.train()
        for batch in torch.randperm(len(train), generator=generator).split(32):
            optimiser.zero_grad()
            error = model(x_train[batch]).squeeze(1)-target[batch]
            loss = (w_train[batch]*error.square()).mean()
            loss.backward()
            optimiser.step()
        model.eval()
        with torch.no_grad():
            train_mse = (w_train*(model(x_train).squeeze(1)-target).square()).mean().item()
            z = model(x_valid).squeeze(1).numpy().astype(float)
        predicted = invert_target(z, transform)
        invalid = int((~np.isfinite(predicted) | (predicted <= 0)).sum())
        score = np.inf if invalid else np.sqrt(np.average(
            (np.log10(predicted)-np.log10(y_valid))**2, weights=w_valid))
        history.append([epoch, train_mse, score, invalid])
        if score < best:
            best, best_state, best_epoch, stale = score, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            stale += 1
        if stale >= 30:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history, columns=[
        'epoch', 'training_transformed_MSE', 'validation_log10_RMSE', 'invalid_validation_predictions'
    ]), best_epoch


In [ ]:
# 8: save every comparison, select by validation only
RUN = PROJECT / 'models/S2_SPM/pretrain_01'
assert not RUN.exists(), 'Use a new RUN folder to preserve earlier comparisons.'
RUN.mkdir(parents=True)
records, histories, validation_predictions = [], [], []
for hidden in [(15, 15), (32, 16)]:
    for transform in ['log10', 'sqrt']:
        name = f'{hidden[0]}_{hidden[1]}_{transform}'
        model, history, best_epoch = pretrain(hidden, transform)
        histories.append(history.assign(candidate=name))
        if best_epoch is None:
            records.append({'candidate': name, 'valid_checkpoint': False})
            continue
        with torch.no_grad():
            prediction = invert_target(model(x_valid).squeeze(1).numpy().astype(float), transform)
        error = prediction-y_valid
        records.append({'candidate': name, 'valid_checkpoint': True, 'best_epoch': best_epoch,
            'stopped_epoch': int(history['epoch'].iloc[-1]),
            'parameters': sum(p.numel() for p in model.parameters()),
            'MAE_mg_l': np.average(abs(error), weights=w_valid),
            'RMSE_mg_l': np.sqrt(np.average(error**2, weights=w_valid)),
            'R2': 1-np.sum(w_valid*error**2)/np.sum(w_valid*(y_valid-np.average(y_valid, weights=w_valid))**2),
            'log10_RMSE': np.sqrt(np.average((np.log10(prediction)-np.log10(y_valid))**2, weights=w_valid))})
        torch.save({'state_dict': model.state_dict(), 'hidden': list(hidden), 'transform': transform,
            'features': features, 'centre': centre.tolist(), 'spread': spread.tolist(),
            'best_epoch': best_epoch}, RUN / (name+'.pt'))
        validation_predictions.append(valid[['GLORIA_ID', 'platform', 'TSS', 'weight']].assign(
            candidate=name, prediction_mg_l=prediction))
        print(name, 'best epoch:', best_epoch, flush=True)

pretraining_results = pd.DataFrame(records).sort_values('log10_RMSE')
pretraining_history = pd.concat(histories, ignore_index=True)
assert pretraining_results['valid_checkpoint'].any()
winner = pretraining_results.loc[pretraining_results['valid_checkpoint'], 'candidate'].iloc[0]
pretraining_results.to_csv(RUN / 'validation_results.csv', index=False)
pretraining_history.to_csv(RUN / 'learning_curves.csv', index=False)
pd.concat(validation_predictions, ignore_index=True).to_csv(RUN / 'validation_predictions.csv', index=False)
source[['GLORIA_ID', 'platform', 'role', 'weight']].to_csv(RUN / 'used_observations.csv', index=False)
pretraining_results.set_index('candidate')[[
    'best_epoch', 'MAE_mg_l', 'RMSE_mg_l', 'R2', 'log10_RMSE']].round(3)


In [ ]:
# 8: compare the same validation metric, not differently scaled training losses
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.5))
for name, curve in pretraining_history.groupby('candidate', sort=False):
    ax.plot(curve['epoch'], curve['validation_log10_RMSE'].replace(np.inf, np.nan), label=name)
ax.set(xlabel='Epoch', ylabel='Validation log10 RMSE (lower is better)')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RUN / 'validation_learning_curves.png', dpi=150)
plt.show()


The GLORIA validation winner is 15–15 with a square-root target at epoch 36, log10-RMSE = 0.446. Training stopped at epoch 66 after 30 epochs without improvement, then reloaded epoch 36. Physical RMSE is 82.5 mg/L and R² is −4.24. The model selected for proportional error still has large absolute errors.

The 32–16 log10 candidate has lower physical RMSE but a worse selection score. We compare both candidates on local transfer in Section 9. These scores describe GLORIA validation, not local satellite predictions or the reserved GLORIA test groups.


## 9. S2-SPM local fine-tuning

We test whether GLORIA pretraining helps predict local SPM. Fine-tuning updates the pretrained weights with 63 local development observations from 37 dates. A matched random-start network learns from the same local observations without pretrained weights.

The checkpoints are 15–15/square root and 32–16/log10. We compare leaving each pretrained model as it is, updating its output layer, and updating all layers at a smaller learning rate. We also test matching random-start models and a faster-learning random-start run. Controls include a constant prediction, the training geometric mean obtained by averaging log-SPM and converting back, and a random forest.

We group dates into three sets, train on two and check the third. The third set also chooses the stopping epoch, so these scores guide tuning and do not provide an independent test. The median best epoch count across rounds sets the final training duration on all development observations.

All networks and this section's RF control use the GLORIA training scaler. Test data do not fit scaling or coefficients. Section 13 instead gives RF raw inputs. This run needs scikit-learn 1.7.2 and saves results in `FINE`, stopping if the folder already exists.


In [ ]:
# 9: load source checkpoints and reserve development dates for validation
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
import joblib

checkpoints = {name: torch.load(PROJECT / 'models/S2_SPM/pretrain_01' / (name+'.pt'),
    map_location='cpu', weights_only=True) for name in ['15_15_sqrt', '32_16_log10']}
fixed_local = pd.read_csv(PREP / 'local_assignments.csv')
dev_ids = fixed_local.loc[fixed_local['sensor'].eq('S2') & fixed_local['target'].eq('SPM')
                          & fixed_local['role'].eq('development'), 'row_id']
tuning_dev = matchups.set_index('row_id').loc[dev_ids].sort_index().reset_index()
tuning_features = checkpoints['15_15_sqrt']['features']
raw_features = ['raw__' + index_columns.get(c, 'Rrs_'+c) for c in tuning_features]
tuning_input = tuning_dev[raw_features].to_numpy()
assert len(tuning_dev) == 63 and np.isfinite(tuning_input).all()
reference = checkpoints['15_15_sqrt']
assert checkpoints['32_16_log10']['features'] == tuning_features
assert checkpoints['32_16_log10']['centre'] == reference['centre']
assert checkpoints['32_16_log10']['spread'] == reference['spread']
tuning_x = torch.tensor((tuning_input-reference['centre'])/reference['spread'], dtype=torch.float32)
tuning_y = tuning_dev['target_value'].to_numpy()
tuning_folds = list(GroupKFold(3, shuffle=True, random_state=42).split(
    tuning_dev, groups=tuning_dev['acquisition_date']))
tuning_dev['fold'] = -1
for fold, (fit, hold) in enumerate(tuning_folds):
    tuning_dev.loc[hold, 'fold'] = fold
assert tuning_dev.groupby('acquisition_date')['fold'].nunique().eq(1).all()
save_fixed(tuning_dev[['row_id', 'acquisition_date', 'fold']], 'local_cv_S2_SPM.csv')
tuning_dev.groupby('fold').agg(observations=('row_id', 'size'), dates=('acquisition_date', 'nunique'))


In [ ]:
# 9: the source scaler stays fixed, only the permitted network weights change
def tuning_metrics(y, prediction):
    invalid = int((~np.isfinite(prediction) | (prediction <= 0)).sum())
    if invalid:
        return {'MAE_mg_l': np.nan, 'RMSE_mg_l': np.nan, 'R2': np.nan,
                'log10_RMSE': np.inf, 'invalid': invalid}
    error = prediction-y
    return {'MAE_mg_l': np.mean(abs(error)), 'RMSE_mg_l': np.sqrt(np.mean(error**2)),
        'R2': 1-np.sum(error**2)/np.sum((y-y.mean())**2),
        'log10_RMSE': np.sqrt(np.mean((np.log10(prediction)-np.log10(y))**2)), 'invalid': 0}

def adapt_local(checkpoint, initialisation, mode, lr, x, y, x_val=None, y_val=None, epochs=300):
    torch.manual_seed(42)
    h = checkpoint['hidden']
    model = torch.nn.Sequential(torch.nn.Linear(len(tuning_features), h[0]), torch.nn.ReLU(),
        torch.nn.Linear(h[0], h[1]), torch.nn.ReLU(), torch.nn.Linear(h[1], 1))
    if initialisation == 'pretrained':
        model.load_state_dict(checkpoint['state_dict'])
    if mode == 'frozen':
        return model, 0, []
    if mode == 'head':
        for parameter in model[:-1].parameters():
            parameter.requires_grad_(False)
    optimiser = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                                  lr=lr, weight_decay=0.0001)
    target = torch.tensor(np.log10(y) if checkpoint['transform']=='log10' else np.sqrt(y),
                          dtype=torch.float32)
    generator = torch.Generator().manual_seed(42)
    best, state, best_epoch, stale, curve = np.inf, None, None, 0, []
    for epoch in range(1, epochs+1):
        model.train()
        for batch in torch.randperm(len(y), generator=generator).split(32):
            optimiser.zero_grad()
            loss = (model(x[batch]).squeeze(1)-target[batch]).square().mean()
            loss.backward()
            optimiser.step()
        if x_val is None:
            continue
        model.eval()
        with torch.no_grad():
            prediction = invert_target(model(x_val).squeeze(1).numpy().astype(float), checkpoint['transform'])
        scores = tuning_metrics(y_val, prediction)
        curve.append([epoch, scores['log10_RMSE'], scores['invalid']])
        if scores['log10_RMSE'] < best:
            best, state, best_epoch, stale = scores['log10_RMSE'], copy.deepcopy(model.state_dict()), epoch, 0
        else:
            stale += 1
        if state is not None and stale >= 30:
            break
    if state is not None:
        model.load_state_dict(state)
    return model, epochs if x_val is None else best_epoch, curve


In [ ]:
# 9: define the bounded comparison before fitting
FINE = PROJECT / 'models/S2_SPM/finetune_01'
assert not FINE.exists(), 'Use a new FINE folder to preserve earlier comparisons.'
FINE.mkdir(parents=True)
settings = []
for checkpoint in checkpoints:
    settings.append([checkpoint+'_pretrained_frozen', checkpoint, 'pretrained', 'frozen', 0.0])
    for initialisation in ['pretrained', 'random']:
        for mode, lr in [('head', 0.001), ('full', 0.0001)]:
            settings.append([checkpoint+'_'+initialisation+'_'+mode, checkpoint, initialisation, mode, lr])
    settings.append([checkpoint+'_random_full_fast', checkpoint, 'random', 'full', 0.001])
settings = pd.DataFrame(settings, columns=['candidate', 'checkpoint', 'initialisation', 'mode', 'lr'])
settings.to_csv(FINE / 'settings.csv', index=False)

fold_records, cv_predictions, fine_curves = [], [], []
for setting in settings.itertuples(index=False):
    checkpoint = checkpoints[setting.checkpoint]
    for fold, (fit, hold) in enumerate(tuning_folds):
        model, best_epoch, curve = adapt_local(checkpoint, setting.initialisation,
            setting.mode, setting.lr, tuning_x[fit], tuning_y[fit], tuning_x[hold], tuning_y[hold])
        with torch.no_grad():
            prediction = invert_target(model(tuning_x[hold]).squeeze(1).numpy().astype(float),
                                        checkpoint['transform'])
        fold_records.append({'candidate': setting.candidate, 'fold': fold,
                             'best_epoch': best_epoch, **tuning_metrics(tuning_y[hold], prediction)})
        cv_predictions.append(tuning_dev.loc[hold, ['row_id', 'acquisition_date', 'target_value']].assign(
            candidate=setting.candidate, fold=fold, prediction_mg_l=prediction))
        fine_curves.extend([[setting.candidate, fold, *row] for row in curve])
    print(setting.candidate, 'done', flush=True)

# Fixed baselines use only each fold's training observations.
for fold, (fit, hold) in enumerate(tuning_folds):
    forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=1)
    forest.fit(tuning_x[fit].numpy(), np.log10(tuning_y[fit]))
    baseline_predictions = {'random_forest_log10': 10.0**forest.predict(tuning_x[hold].numpy()),
        'constant_log_mean': np.full(len(hold), 10.0**np.log10(tuning_y[fit]).mean())}
    for name, prediction in baseline_predictions.items():
        fold_records.append({'candidate': name, 'fold': fold, 'best_epoch': np.nan,
                             **tuning_metrics(tuning_y[hold], prediction)})
        cv_predictions.append(tuning_dev.loc[hold, ['row_id', 'acquisition_date', 'target_value']].assign(
            candidate=name, fold=fold, prediction_mg_l=prediction))

fold_results = pd.DataFrame(fold_records)
cv_predictions = pd.concat(cv_predictions, ignore_index=True)
fine_curves = pd.DataFrame(fine_curves, columns=[
    'candidate', 'fold', 'epoch', 'validation_log10_RMSE', 'invalid'])
fold_results.to_csv(FINE / 'fold_results.csv', index=False)
cv_predictions.to_csv(FINE / 'cv_predictions.csv', index=False)
fine_curves.to_csv(FINE / 'learning_curves.csv', index=False)


In [ ]:
# 9: pooled development predictions, all 63 rows must be present
local_results = []
for name, part in cv_predictions.groupby('candidate', sort=False):
    assert len(part) == 63 and part['row_id'].is_unique
    local_results.append({'candidate': name,
        **tuning_metrics(part['target_value'].to_numpy(), part['prediction_mg_l'].to_numpy())})
local_results = pd.DataFrame(local_results).sort_values('log10_RMSE')
local_results.to_csv(FINE / 'cv_results.csv', index=False)
local_results.set_index('candidate').round(3)


In [ ]:
# 9: refit selected neural versions on development only, with no test monitoring
final_choices = {}
for initialisation in ['pretrained', 'random']:
    eligible = local_results.merge(settings, on='candidate')
    eligible = eligible.loc[eligible['initialisation'].eq(initialisation)
                            & np.isfinite(eligible['log10_RMSE'])].sort_values('log10_RMSE')
    assert len(eligible), 'No valid neural configuration, inspect failures before refitting.'
    chosen = eligible.iloc[0]
    best_epochs = fold_results.loc[fold_results['candidate'].eq(chosen['candidate']), 'best_epoch']
    epochs = int(np.median(best_epochs))
    checkpoint = checkpoints[chosen['checkpoint']]
    model, _, _ = adapt_local(checkpoint, initialisation, chosen['mode'], chosen['lr'],
                              tuning_x, tuning_y, epochs=epochs)
    torch.save({'state_dict': model.state_dict(), 'hidden': checkpoint['hidden'],
        'features': checkpoint['features'], 'transform': checkpoint['transform'],
        'centre': checkpoint['centre'], 'spread': checkpoint['spread'],
        'source_candidate': chosen['checkpoint'], 'local_candidate': chosen['candidate'],
        'local_epochs': epochs, 'mode': chosen['mode'], 'lr': float(chosen['lr'])},
        FINE / ('final_'+initialisation+'.pt'))
    final_choices[initialisation] = {'candidate': chosen['candidate'], 'development_epochs': epochs}

forest = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=1)
forest.fit(tuning_x.numpy(), np.log10(tuning_y))
joblib.dump({'model': forest, 'features': tuning_features, 'centre': reference['centre'],
    'spread': reference['spread'], 'transform': 'log10'}, FINE / 'final_random_forest.joblib')
pd.DataFrame(final_choices).T


The best development log10-RMSE is 0.449 for the 32–16 log10 pretrained model with all layers updated. The best random-start model scores 0.480 and RF scores 0.469. RF has lower physical RMSE, 32.8 versus 35.6 mg/L. The fine-tuned network's R² is only 0.082. Its advantage depends on the measure and is small.

We train the selected pretrained and random-start models on all 63 development observations for 33 and 140 epochs respectively. RF uses the same development set. Section 10 evaluates the saved models on 17 reserved observations from nine dates.


## 10. S2-SPM reserved test dates

The pretrained MLP, random-start MLP and RF predict the same 17 reserved observations from nine dates. Fitting and tuning excluded these dates, although the initial correlations included them. We compare predictions with field SPM in mg/L using every reserved observation.

We do not refit or choose settings from these errors. The test has now been viewed, so later development work must account for this exposure, even when its test rows are excluded from fitting.


In [ ]:
# 10: load the selected models, no fitting or new scaling
EVAL = PROJECT / 'models/S2_SPM/test_01'
assert not EVAL.exists(), 'Saved test results already exist. Read them instead of overwriting.'
FINE = PROJECT / 'models/S2_SPM/finetune_01'
saved_networks = {name: torch.load(FINE / filename, map_location='cpu', weights_only=True)
    for name, filename in [('Pretrained MLP', 'final_pretrained.pt'), ('Random-start MLP', 'final_random.pt')]}
saved_forest = joblib.load(FINE / 'final_random_forest.joblib')
roles = pd.read_csv(DATA / 'model_preparation/local_assignments.csv')
test_ids = roles.loc[roles['sensor'].eq('S2') & roles['target'].eq('SPM') & roles['role'].eq('test'), 'row_id']
test_rows = matchups.set_index('row_id').loc[test_ids].sort_index().reset_index()
assert len(test_rows) == 17 and test_rows['acquisition_date'].nunique() == 9
saved_scaler = saved_networks['Pretrained MLP']
for saved in [saved_networks['Random-start MLP'], saved_forest]:
    for key in ['features', 'centre', 'spread']:
        assert saved[key] == saved_scaler[key]
columns = ['raw__' + index_columns.get(c, 'Rrs_'+c) for c in saved_scaler['features']]
x_test = torch.tensor((test_rows[columns].to_numpy()-saved_scaler['centre'])/saved_scaler['spread'], dtype=torch.float32)
assert torch.isfinite(x_test).all()
test_predictions = test_rows[['row_id', 'sample_id', 'station_id', 'acquisition_date',
                              'datetime_utc', 'acquisition_start_utc', 'target_value', 'target_unit']].copy()
for name, saved in saved_networks.items():
    h = saved['hidden']
    network = torch.nn.Sequential(torch.nn.Linear(len(columns), h[0]), torch.nn.ReLU(),
        torch.nn.Linear(h[0], h[1]), torch.nn.ReLU(), torch.nn.Linear(h[1], 1))
    network.load_state_dict(saved['state_dict'])
    network.eval()
    with torch.no_grad():
        z = network(x_test).squeeze(1).numpy().astype(float)
    test_predictions[name] = invert_target(z, saved['transform'])
test_predictions['Random forest'] = 10.0**saved_forest['model'].predict(x_test.numpy())
EVAL.mkdir(parents=True)
test_predictions.to_csv(EVAL / 'test_predictions.csv', index=False)


In [ ]:
# 10: same metrics and every reserved observation for all three models
model_names = ['Pretrained MLP', 'Random-start MLP', 'Random forest']
test_results = pd.DataFrame([{'model': name, 'n': len(test_rows),
    'dates': test_rows['acquisition_date'].nunique(),
    **tuning_metrics(test_rows['target_value'].to_numpy(), test_predictions[name].to_numpy())}
    for name in model_names])
test_results.to_csv(EVAL / 'test_results.csv', index=False)
test_results.set_index('model').round(3)


In [ ]:
# 10: observed versus predicted SPM, same axes for each model
fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharex=True, sharey=True)
upper = max(test_predictions[['target_value']+model_names].max())*1.05
for ax, name in zip(axes, model_names):
    ax.scatter(test_predictions['target_value'], test_predictions[name], s=25)
    ax.plot([0, upper], [0, upper], color='grey', linestyle='--', linewidth=1)
    ax.set(title=name, xlabel='Observed SPM (mg/L)', xlim=(0, upper), ylim=(0, upper))
axes[0].set_ylabel('Predicted SPM (mg/L)')
fig.tight_layout()
fig.savefig(EVAL / 'observed_predicted.png', dpi=150)
plt.show()


Pretraining did not improve this S2-SPM test. The random-start MLP gives log10-RMSE = 0.364 and physical RMSE = 33.6 mg/L, with RF close in physical RMSE. The pretrained model gives 0.408 and 37.1 mg/L, with R² = 0.024.

All three models underestimate the two highest measurements. The dashed line marks perfect agreement, and points below it are underestimates. With 17 observations from nine dates, the comparison gives limited evidence about their ranking. We next tested simpler models and more data, without tuning against these test measurements.


## 11. GLORIA satellite-matchup feasibility

The pretrained model's S2-SPM test RMSE was 37.1 mg/L, compared with 33.6 mg/L for the random-start network. There were just 63 local development observations. Limited data could contribute to the weak results, although this comparison cannot establish the cause.

We tried expanding the training data by matching GLORIA field SPM/Secchi measurements to actual satellite reflectance, with the same processing as the regional data. Models would then learn from satellite inputs throughout. An observation could be useful even if its measured spectrum was unsuitable for pretraining.

We first searched image metadata. Candidates needed a positive label, coordinates, a declared UTC time, a location inside an image footprint and an acquisition interval within three hours. This finds possible coverage. Cloud and pixel-quality checks follow download. The table shows saved catalogue counts, not approved training samples. [Search record](data/gloria_catalogue/run.json).


In [ ]:
# 11: saved availability, not new training observations
catalogue_counts = pd.read_csv(DATA / 'gloria_catalogue/summary.csv', dtype={'collection': str})
catalogue_counts.loc[catalogue_counts['collection'].ne('004'), [
    'sensor', 'collection', 'target', 'candidate_observations',
    'observations_with_3h_candidate', 'field_dates', 'reported_sites']]


The search found extra SPM coverage, including 49 S2 and 65 S3 candidate records from Hawke Bay with repeated 00:00 UTC timestamps. We could not confirm whether these were midnight measurements or missing time information, so they were set aside for review.

Without those records, there were 48 S2 and 82 S3 SPM candidates before pixel-quality checks. Secchi added few dates. The catalogue search downloaded no imagery. We then retrieved and processed the candidate images using the regional rules. The next table shows the usable observations after excluding uncertain times. [Per-sample outcomes](gloria_satellite/sample_status.csv).


In [ ]:
# 11: actual matchups, with unresolved timestamps kept separate
satellite_gloria = pd.read_csv(PROJECT / 'gloria_satellite/matchups.csv')
gloria_ready = satellite_gloria.loc[~satellite_gloria['time_precision_review']].copy()
cohorts = pd.MultiIndex.from_product([['S2', 'S3'], ['SPM', 'Secchi']], names=['sensor', 'target'])
gloria_ready.groupby(['sensor', 'target']).agg(
    observations=('sample_id', 'nunique'), dates=('acquisition_date', 'nunique'),
    areas=('station_id', 'nunique')
).reindex(cohorts, fill_value=0)


All 67 products were processed and the saved cutouts passed verification. They added few usable observations. Five S2 Secchi observations came from one date, and no S3 Secchi window passed the primary screen. Another 29 S2-SPM and 14 S3-SPM rows passed satellite quality checks but had unresolved times. [Checks and provenance](gloria_satellite/completion_checks.json).

These counts describe satellite and time eligibility. Section 16 further restricts training additions to complete-input observations from GLORIA's training groups. Sections 13–15 use no additions. This expansion uses satellite measurements, unlike the field-spectral pretraining in Sections 8–10.

We use coastal/estuarine sites, separate labels, the three-hour limit, and at least 20/25 valid S2 or 9/9 valid S3 pixels. Some native quality flags exclude negative reflectance, but we do not clip accepted values to zero. Variation within windows is not an extra filter.

A 5/9 S3 valid-pixel rule would give 13 time-eligible SPM and three Secchi observations, compared with ten and zero. We calculated this as a coverage check but did not adopt it. Unresolved Hawke Bay times are excluded from training and hourly wind matching.


## 12. ERA5 wind

We also tested wind as extra information about regional conditions. Wind can affect circulation, sediment resuspension and transport. [Baeye et al. (2011)](https://doi.org/10.1007/s10236-010-0370-7) associated NE winds with higher SPM and SW winds with lower SPM on the Belgian inner shelf. We therefore included direction as well as speed, without assuming the same response at every station.

Hourly wind at 10 m comes from [ERA5](https://doi.org/10.24381/cds.adbb2d47) through [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api). We use the nearest 0.25° cell, UTC and m/s, without elevation adjustment. These are regional estimates, not point measurements. `dataset_origin` distinguishes local from GLORIA-derived satellite pairs.

For each overpass, we use the latest hour at or before the native acquisition time. Means cover the last 6, 12 or 24 hourly values ending then. We use no later values or filled gaps. The code reconstructs features from saved hours and checks them against the feature table, without another download.

We convert direction to eastward $u$ and northward $v$ velocity before averaging. Averaging angles directly would mishandle directions around 0°/360°. The negative signs follow the meteorological “wind from” convention ([ECMWF](https://confluence.ecmwf.int/pages/viewpage.action?pageId=133262398)). Observations with unresolved times get no weather values. We chose these history lengths for the experiment. Wind adds inputs rather than observations, and using reanalysis means this is retrospective analysis. [Requests and provenance](weather/run.json).


In [ ]:
# 12: saved hourly data and matchup identifiers, no new download
weather_samples = pd.read_csv(PROJECT / 'weather/samples.csv')
wind_hourly = pd.read_csv(PROJECT / 'weather/hourly_wind.csv')
wind_hourly['time_utc'] = pd.to_datetime(wind_hourly['time_utc'], utc=True)
weather_samples['wind_hour_utc'] = pd.to_datetime(weather_samples['wind_hour_utc'], utc=True)
angle = np.deg2rad(wind_hourly['wind_from_deg'])
wind_hourly['wind_u_ms'] = -wind_hourly['wind_speed_ms'] * np.sin(angle)
wind_hourly['wind_v_ms'] = -wind_hourly['wind_speed_ms'] * np.cos(angle)
wind_hourly = wind_hourly.set_index(['cell_id', 'time_utc']).sort_index()


In [ ]:
# 12: reconstruct wind features from the saved hours
wind_records = []
eligible_weather = weather_samples.loc[~weather_samples['time_precision_review']]
for row in eligible_weather.drop_duplicates('weather_id').itertuples():
    times = pd.date_range(row.wind_hour_utc - pd.Timedelta(hours=23), row.wind_hour_utc, freq='h')
    history = wind_hourly.loc[row.cell_id].reindex(times)
    assert history[['wind_speed_ms', 'wind_u_ms', 'wind_v_ms']].notna().all().all()
    record = {'weather_id': row.weather_id}
    for n in [1, 6, 12, 24]:
        for name in ['speed', 'u', 'v']:
            record[f'wind_{name}_{n}h_ms'] = history[f'wind_{name}_ms'].tail(n).mean()
        u, v = record[f'wind_u_{n}h_ms'], record[f'wind_v_{n}h_ms']
        record[f'wind_from_{n}h_deg'] = (np.degrees(np.arctan2(-u, -v)) % 360
            if np.hypot(u, v) > 1e-6 else np.nan)
    wind_records.append(record)
wind_features = pd.DataFrame(wind_records)
saved_wind = pd.read_csv(PROJECT / 'weather/wind_features.csv')[wind_features.columns]
pd.testing.assert_frame_equal(wind_features, saved_wind, check_exact=False, rtol=1e-10, atol=1e-10)


In [ ]:
# 12: primary collections only, with uncertain times shown separately
weather_matchups = pd.read_csv(PROJECT / 'weather/matchups_wind.csv', dtype={'collection': str})
weather_matchups.loc[weather_matchups['primary_collection']].groupby(
    ['dataset_origin', 'sensor', 'target', 'weather_status']).agg(
    observations=('row_id', 'size'), dates=('acquisition_date', 'nunique'))


## 13. Local-only baselines

We compare simpler models on local satellite–field pairs, separately for the four sensor/target datasets. Within each dataset, every model uses the same complete-input observations and date groups. Wind and GLORIA satellite additions are excluded.

The first model is a power law, $\widehat{y}=A x^b$, where $y$ is SPM or Secchi and $x$ is red-edge reflectance. $A$ sets the scale and $b$ controls the response. Taking logs gives the straight line `log10(y) = log10(A) + b × log10(x)`. Least squares estimates the two coefficients from each training set. We chose red edge as an exploratory input, without copying a published retrieval equation.

| Model | How it predicts | Inputs and settings |
| --- | --- | --- |
| Power law | Fits the log–log straight line above | Positive red-edge reflectance only |
| Random forest (RF) | Averages 200 decision trees, each splitting observations by input values | Eight bands/indices, minimum three samples per final leaf, seed 42 |
| Support-vector regression (SVR) | Fits a flexible nonlinear relationship using a radial-basis-function kernel | Eight inputs, C = 1/10/100 and epsilon = 0.05/0.1 |
| Constant | Predicts the training geometric mean for every observation | Checks whether the optical inputs help |

We use the RF and SVR model families described by [Breiman (2001)](https://doi.org/10.1023/A:1010933404324) and [Smola & Schölkopf (2004)](https://doi.org/10.1023/B:STCO.0000035301.49549.88), with settings chosen for this comparison. All models learn log10(target), and predictions are converted back to mg/L or metres without a log-bias correction. RF uses unscaled inputs. SVR scaling uses the medians and IQRs of its training set.

In each of three rounds, we put aside one date group for evaluation. We use three smaller date-grouped checks within the other observations to choose SVR settings, then train on those observations and predict the excluded group. Evaluation dates therefore do not choose scaling or settings. The code calls these evaluation and tuning splits `outer` and `inner` folds. [Scikit-learn nested validation](https://scikit-learn.org/1.7/auto_examples/model_selection/plot_nested_cross_validation_iris.html), [grouped-data guidance](https://doi.org/10.1111/ecog.02881).

The comparison uses 63 S2-SPM, 44 S2-Secchi, 104 S3-SPM and 59 S3-Secchi observations. Five incomplete S3 rows are recorded in the source tables but excluded equally from the models. We predict development observations only. Section 17 tests further literature-based regressions on the same rows.


In [ ]:
# 13: development data only, existing features and S2-SPM folds
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.dummy import DummyRegressor

BASE = PROJECT / 'models/local_baselines_01'
assert not BASE.exists(), 'Read the saved run or use a new folder.'
BASE.mkdir(parents=True)
baseline_roles = pd.read_csv(PREP / 'local_assignments.csv')
baseline_ids = baseline_roles.loc[baseline_roles['role'].eq('development'), 'row_id']
baseline_dev = matchups.set_index('row_id').loc[baseline_ids].sort_index().reset_index()
baseline_cohorts, fold_records, baseline_counts = {}, [], []
for (sensor, target), group in baseline_dev.groupby(['sensor', 'target']):
    group = group.reset_index(drop=True)
    columns = ['raw__' + index_columns.get(f, 'Rrs_'+f) for f in selected_inputs[sensor, target]]
    if (sensor, target) == ('S2', 'SPM'):
        saved = pd.read_csv(PREP / 'local_cv_S2_SPM.csv').set_index('row_id')
        group['fold'] = saved.loc[group['row_id'], 'fold'].to_numpy()
    else:
        group['fold'] = -1
        for fold, (_, hold) in enumerate(GroupKFold(3, shuffle=True, random_state=42).split(
                group, groups=group['acquisition_date'])):
            group.loc[hold, 'fold'] = fold
    group['input_complete'] = np.isfinite(group[columns]).all(axis=1)
    fold_records.append(group[['row_id', 'sensor', 'target', 'acquisition_date', 'fold', 'input_complete']])
    complete = group.loc[group['input_complete']].reset_index(drop=True)
    baseline_counts.append([sensor, target, len(group), len(complete),
                            complete['acquisition_date'].nunique(), len(group)-len(complete)])
    baseline_cohorts[sensor, target] = (complete, columns)
baseline_folds = pd.concat(fold_records, ignore_index=True)
baseline_folds.to_csv(BASE / 'fold_assignments.csv', index=False)
baseline_counts = pd.DataFrame(baseline_counts, columns=[
    'sensor', 'target', 'development_rows', 'complete_rows', 'dates', 'incomplete'])
baseline_counts.to_csv(BASE / 'counts.csv', index=False)
baseline_counts


In [ ]:
# 13: held-out development predictions, SVR tuning stays inside each training set
baseline_predictions, svr_choices, power_coefficients = [], [], []
for (sensor, target), (group, columns) in baseline_cohorts.items():
    edge = 'raw__Rrs_' + ('B5' if sensor == 'S2' else 'Oa11')
    for fold in range(3):
        train, hold = group.loc[group['fold'].ne(fold)], group.loc[group['fold'].eq(fold)]
        assert set(train['acquisition_date']).isdisjoint(hold['acquisition_date'])
        log_y = np.log10(train['target_value'].to_numpy())
        models = {
            'Constant': DummyRegressor(strategy='mean'),
            'Power law': make_pipeline(FunctionTransformer(np.log10), LinearRegression()),
            'Random forest': RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                                                   random_state=42, n_jobs=1),
            'SVR': GridSearchCV(make_pipeline(RobustScaler(), SVR(kernel='rbf', gamma='scale')),
                {'svr__C': [1, 10, 100], 'svr__epsilon': [0.05, 0.1]},
                cv=GroupKFold(3, shuffle=True, random_state=42),
                scoring='neg_mean_squared_error', n_jobs=1, error_score='raise')}
        for name, model in models.items():
            inputs = [edge] if name == 'Power law' else columns
            if name == 'Power law':
                assert train[edge].gt(0).all() and hold[edge].gt(0).all()
            fit_options = {'groups': train['acquisition_date']} if name == 'SVR' else {}
            model.fit(train[inputs].to_numpy(), log_y, **fit_options)
            prediction = 10.0**model.predict(hold[inputs].to_numpy())
            assert np.isfinite(prediction).all() and (prediction > 0).all()
            baseline_predictions.append(hold[['row_id', 'sensor', 'target', 'target_unit',
                'acquisition_date', 'fold', 'target_value']].assign(model=name, prediction=prediction))
            if name == 'SVR':
                svr_choices.append({'sensor': sensor, 'target': target, 'fold': fold, **model.best_params_})
            if name == 'Power law':
                linear = model.named_steps['linearregression']
                power_coefficients.append({'sensor': sensor, 'target': target, 'fold': fold,
                    'a': 10.0**linear.intercept_, 'b': linear.coef_[0], 'input': edge})
    print(sensor, target, 'done')
baseline_predictions = pd.concat(baseline_predictions, ignore_index=True)
baseline_predictions.to_csv(BASE / 'development_predictions.csv', index=False)
pd.DataFrame(svr_choices).to_csv(BASE / 'svr_choices.csv', index=False)
pd.DataFrame(power_coefficients).to_csv(BASE / 'power_coefficients.csv', index=False)


In [ ]:
# 13: pooled held-out development errors, units differ between targets
baseline_results = []
for (sensor, target, unit, model), group in baseline_predictions.groupby(
        ['sensor', 'target', 'target_unit', 'model'], sort=False):
    observed, predicted = group['target_value'].to_numpy(), group['prediction'].to_numpy()
    error = predicted-observed
    baseline_results.append({'sensor': sensor, 'target': target, 'unit': unit, 'model': model,
        'n': len(group), 'dates': group['acquisition_date'].nunique(),
        'MAE': np.mean(abs(error)), 'RMSE': np.sqrt(np.mean(error**2)),
        'R2': 1-np.sum(error**2)/np.sum((observed-observed.mean())**2),
        'log10_RMSE': np.sqrt(np.mean((np.log10(predicted)-np.log10(observed))**2))})
baseline_results = pd.DataFrame(baseline_results)
baseline_results.to_csv(BASE / 'development_results.csv', index=False)
baseline_results.set_index(['sensor', 'target', 'unit', 'model']).round(3)


RF gives development R² = 0.223 for S2-SPM, 0.476 for S2-Secchi, 0.543 for S3-SPM and 0.586 for S3-Secchi. For S2-SPM, the red-edge power law has lower log10-RMSE, 0.440 versus 0.469, but RF has lower physical RMSE, 32.8 versus 33.5 mg/L. The preferred model depends on the error measure.

The S3 Secchi power law predicts 42.3 m for a measured 2.6 m, giving worse RMSE than the constant baseline. We include this extreme prediction in the errors. It shows how badly the curve can extrapolate.

These are development scores. The S2-SPM neural experiment used the same evaluation dates but also used their labels to choose stopping epochs, so its scores involved more tuning on those observations. We did not train the other neural sensor/target versions. This comparison uses neither wind nor GLORIA satellite additions. [Run and checks](models/local_baselines_01/run.json).


## 14. Does wind help?

We compare optical inputs, wind inputs, and both together. “Reflectance” in the tables includes the three bands and five indices. We use Section 13's RF/SVR settings, observations and date groups to isolate the effect of adding wind.

The six wind inputs are latest hourly speed, mean speed over 6/12/24 hours, and mean eastward/northward velocity over 24 hours. Speed describes strength and recent history. The velocity components describe net direction over the day without averaging angles. We chose this compact set before the comparison, without searching for the best lags.

We reuse the optical-only predictions. RF uses the same settings. SVR repeats its six-setting search and scaling within each round's training dates. Models learn log10(target), and we also report physical-unit errors. The comparison excludes GLORIA additions and test predictions. Adding wind does not increase the number of observations.


In [ ]:
# 14: join wind by the exact matchup ID and reuse saved development folds
WIND_RUN = PROJECT / 'models/wind_comparison_01'
assert not WIND_RUN.exists(), 'Read the saved run or use a new folder.'
WIND_RUN.mkdir(parents=True)
wind_inputs = ['wind_speed_1h_ms', 'wind_speed_6h_ms', 'wind_speed_12h_ms',
               'wind_speed_24h_ms', 'wind_u_24h_ms', 'wind_v_24h_ms']
wind_folds = pd.read_csv(PROJECT / 'models/local_baselines_01/fold_assignments.csv')
wind_ids = wind_folds.loc[wind_folds['input_complete'], 'row_id']
wind_dev = matchups.set_index('row_id').loc[wind_ids].reset_index()
wind_dev = wind_dev.merge(wind_folds[['row_id', 'fold']], on='row_id', validate='one_to_one')
wind_table = pd.read_csv(PROJECT / 'weather/matchups_wind.csv')
wind_table = wind_table.loc[wind_table['dataset_origin'].eq('local') & wind_table['weather_status'].eq('complete')]
wind_dev = wind_dev.merge(wind_table[['row_id', 'cell_id', 'wind_hour_utc']+wind_inputs],
                         on='row_id', how='left', validate='one_to_one')
assert len(wind_dev) == len(wind_ids) and np.isfinite(wind_dev[wind_inputs]).all().all()
wind_dev['wind_hour_utc'] = pd.to_datetime(wind_dev['wind_hour_utc'], utc=True)
wind_dev[['row_id', 'sensor', 'target', 'acquisition_date', 'fold', 'cell_id',
          'wind_hour_utc']+wind_inputs].to_csv(WIND_RUN / 'development_inputs.csv', index=False)
wind_dev.groupby(['sensor', 'target']).agg(
    observations=('row_id', 'size'), dates=('acquisition_date', 'nunique'))


In [ ]:
# 14: same rows and folds, change only the input set
previous = pd.read_csv(PROJECT / 'models/local_baselines_01/development_predictions.csv')
wind_predictions = [previous.loc[previous['model'].isin(['Random forest', 'SVR'])].assign(inputs='Reflectance')]
wind_choices, wind_inner_scores = [], []
for (sensor, target), group in wind_dev.groupby(['sensor', 'target']):
    spectral = ['raw__' + index_columns.get(f, 'Rrs_'+f) for f in selected_inputs[sensor, target]]
    for inputs, columns in [('Wind', wind_inputs), ('Reflectance + wind', spectral+wind_inputs)]:
        for fold in range(3):
            train, hold = group.loc[group['fold'].ne(fold)], group.loc[group['fold'].eq(fold)]
            assert set(train['acquisition_date']).isdisjoint(hold['acquisition_date'])
            models = {
                'Random forest': RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                                                       random_state=42, n_jobs=1),
                'SVR': GridSearchCV(make_pipeline(RobustScaler(), SVR(kernel='rbf', gamma='scale')),
                    {'svr__C': [1, 10, 100], 'svr__epsilon': [0.05, 0.1]},
                    cv=GroupKFold(3, shuffle=True, random_state=42),
                    scoring='neg_mean_squared_error', n_jobs=1, error_score='raise')}
            for name, model in models.items():
                options = {'groups': train['acquisition_date']} if name == 'SVR' else {}
                model.fit(train[columns].to_numpy(), np.log10(train['target_value']), **options)
                prediction = 10.0**model.predict(hold[columns].to_numpy())
                assert np.isfinite(prediction).all() and (prediction > 0).all()
                wind_predictions.append(hold[['row_id', 'sensor', 'target', 'target_unit',
                    'acquisition_date', 'fold', 'target_value']].assign(
                    model=name, inputs=inputs, prediction=prediction))
                if name == 'SVR':
                    wind_choices.append({'sensor': sensor, 'target': target, 'inputs': inputs,
                                         'fold': fold, **model.best_params_})
                    scores = pd.DataFrame(model.cv_results_)
                    wind_inner_scores.append(scores[['param_svr__C', 'param_svr__epsilon',
                        'mean_test_score', 'rank_test_score']].assign(
                        sensor=sensor, target=target, inputs=inputs, fold=fold))
    print(sensor, target, 'done')
wind_predictions = pd.concat(wind_predictions, ignore_index=True)
wind_predictions.to_csv(WIND_RUN / 'development_predictions.csv', index=False)
pd.DataFrame(wind_choices).to_csv(WIND_RUN / 'svr_choices.csv', index=False)
pd.concat(wind_inner_scores, ignore_index=True).to_csv(WIND_RUN / 'inner_scores.csv', index=False)


In [ ]:
# 14: paired development errors, the same observations in every input set
wind_results = []
for (sensor, target, unit, model, inputs), group in wind_predictions.groupby(
        ['sensor', 'target', 'target_unit', 'model', 'inputs'], sort=False):
    observed, predicted = group['target_value'].to_numpy(), group['prediction'].to_numpy()
    error = predicted-observed
    wind_results.append({'sensor': sensor, 'target': target, 'unit': unit,
        'model': model, 'inputs': inputs, 'n': len(group), 'dates': group['acquisition_date'].nunique(),
        'MAE': np.mean(abs(error)), 'RMSE': np.sqrt(np.mean(error**2)),
        'R2': 1-np.sum(error**2)/np.sum((observed-observed.mean())**2),
        'log10_RMSE': np.sqrt(np.mean((np.log10(predicted)-np.log10(observed))**2))})
wind_results = pd.DataFrame(wind_results)
wind_results.to_csv(WIND_RUN / 'development_results.csv', index=False)
wind_summary = wind_results.pivot(index=['sensor', 'target', 'unit', 'model'],
    columns='inputs', values=['log10_RMSE', 'RMSE', 'R2'])
wind_summary = wind_summary.reindex(columns=pd.MultiIndex.from_product(
    [['log10_RMSE', 'RMSE', 'R2'], ['Reflectance', 'Wind', 'Reflectance + wind']]))
wind_summary.round(3)


Adding wind gives RF small log-error reductions, with little change in physical RMSE: 32.76 to 33.14 mg/L for S2-SPM, 1.329 to 1.314 m for S2-Secchi, 17.53 to 17.54 mg/L for S3-SPM, and 1.325 to 1.335 m for S3-Secchi. SVR worsens in all four datasets. Wind-only R² is negative throughout.

We chose optical-only inputs after seeing little gain from these six wind features. This result applies to the summaries, samples and models tested. It does not rule out physical wind effects, and we have not quantified uncertainty in the score differences. The comparison uses development data without GLORIA satellite additions. [Run and checks](models/wind_comparison_01/run.json).


## 15. S3 centre pixel

A field sample represents a point, while a 3 × 3 S3 window spans about 900 m. We test the nearest nominal 300 m pixel as a smaller footprint. It may better represent the sample location but can also be noisier.

We use the same C003 images, 9/9 window-quality rule and date groups. This changes the reflectance footprint without admitting additional images. Cutout metadata identifies the centre pixel. We convert native water reflectance to $R_{rs}$ with the verified factor 1/π and rebuild the five indices with the same positive-input safeguards.

Two centre observations have a missing index. We train both footprint alternatives on the intersection of complete-input rows, then compare RF/SVR with and without the six wind features. The source table includes the excluded rows. This experiment changes no S2 data and uses no GLORIA additions or test predictions.

Repeating the extraction needs the cutouts and receipts at `cutout_absolute` and `source_receipt`. Extracted features and results are saved in `models/s3_centre_01`.


In [ ]:
# 15: read the recorded native centre and keep the original selection
import json
from netCDF4 import Dataset
from olci_native import decoded, as_uint, flag_definitions, mask_flags

CENTRE = PROJECT / 'models/s3_centre_01'
assert not CENTRE.exists(), 'Read the saved run or use a new folder.'
CENTRE.mkdir(parents=True)
centre_folds = pd.read_csv(PROJECT / 'models/local_baselines_01/fold_assignments.csv')
centre_folds = centre_folds.loc[centre_folds['sensor'].eq('S3') & centre_folds['input_complete']]
centre_dev = matchups.set_index('row_id').loc[centre_folds['row_id']].reset_index()
centre_dev = centre_dev.merge(centre_folds[['row_id', 'fold']], on='row_id', validate='one_to_one')
centre_records = []
for row in centre_dev.itertuples():
    receipt = json.loads(Path(row.source_receipt).read_text())
    with Dataset(row.cutout_absolute) as nc:
        assert nc.source_product_id == row.product_id == receipt['product_id']
        i, j = int(nc.central_row_in_cutout), int(nc.central_column_in_cutout)
        selection = (slice(i-1, i+2), slice(j-1, j+2))
        flags = nc['WQSF']
        good, _ = mask_flags(as_uint(flags, selection), flag_definitions(flags), '003')
        assert good.shape == (3, 3) and good.all()
        record = {'row_id': row.row_id, 'centre_row': i, 'centre_column': j}
        for band in ['Oa06', 'Oa08', 'Oa11']:
            factor = receipt['metadata'][band]['rrs_factor']
            assert np.isclose(factor, 1/np.pi)
            record['centre__Rrs_'+band] = float(decoded(nc[band+'_reflectance'], (i, j)))*factor
        centre_records.append(record)
centre_values = pd.DataFrame(centre_records)
g, r, e = [centre_values['centre__Rrs_'+b] for b in ['Oa06', 'Oa08', 'Oa11']]
for name, a, b, nd in [('NDTI', r, g, True), ('red_green_ratio', r, g, False),
        ('rededge_red_ratio', e, r, False), ('rededge_green_ratio', e, g, False),
        ('ND_rededge_red', e, r, True)]:
    centre_values['centre__'+name] = ((a-b)/(a+b) if nd else a/b).where((a>1e-6) & (b>1e-6))
centre_columns = [c for c in centre_values if c.startswith('centre__')]
centre_values['centre_complete'] = np.isfinite(centre_values[centre_columns]).all(axis=1)
centre_values.to_csv(CENTRE / 'centre_features.csv', index=False)
centre_dev = centre_dev.merge(centre_values, on='row_id', validate='one_to_one')
centre_counts = centre_dev.groupby('target').agg(
    window_complete=('row_id', 'size'), paired_complete=('centre_complete', 'sum'))
centre_counts['centre_incomplete'] = centre_counts['window_complete']-centre_counts['paired_complete']
centre_counts.to_csv(CENTRE / 'counts.csv')
centre_paired = centre_dev.loc[centre_dev['centre_complete']].copy()
centre_weather = pd.read_csv(PROJECT / 'weather/matchups_wind.csv')
centre_paired = centre_paired.merge(centre_weather[['row_id']+wind_inputs], on='row_id',
                                  how='left', validate='one_to_one')
assert np.isfinite(centre_paired[wind_inputs]).all().all()
centre_paired[['row_id', 'target', 'acquisition_date', 'fold']].to_csv(CENTRE / 'paired_rows.csv', index=False)
centre_counts


In [ ]:
# 15: refit both footprints on identical rows, with the same optional wind inputs
centre_predictions, centre_choices = [], []
for target, group in centre_paired.groupby('target'):
    suffixes = [index_columns.get(f, 'Rrs_'+f) for f in selected_inputs['S3', target]]
    for footprint, prefix in [('3x3 mean', 'raw__'), ('Centre', 'centre__')]:
        for wind in ['No', 'Yes']:
            columns = [prefix+c for c in suffixes] + (wind_inputs if wind == 'Yes' else [])
            for fold in range(3):
                train, hold = group.loc[group['fold'].ne(fold)], group.loc[group['fold'].eq(fold)]
                assert set(train['acquisition_date']).isdisjoint(hold['acquisition_date'])
                models = {
                    'Random forest': RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                                                           random_state=42, n_jobs=1),
                    'SVR': GridSearchCV(make_pipeline(RobustScaler(), SVR(kernel='rbf', gamma='scale')),
                        {'svr__C': [1, 10, 100], 'svr__epsilon': [0.05, 0.1]},
                        cv=GroupKFold(3, shuffle=True, random_state=42),
                        scoring='neg_mean_squared_error', n_jobs=1, error_score='raise')}
                for name, model in models.items():
                    options = {'groups': train['acquisition_date']} if name == 'SVR' else {}
                    model.fit(train[columns].to_numpy(), np.log10(train['target_value']), **options)
                    prediction = 10.0**model.predict(hold[columns].to_numpy())
                    assert np.isfinite(prediction).all() and (prediction > 0).all()
                    centre_predictions.append(hold[['row_id', 'target', 'target_unit',
                        'acquisition_date', 'fold', 'target_value']].assign(
                        footprint=footprint, wind=wind, model=name, prediction=prediction))
                    if name == 'SVR':
                        centre_choices.append({'target': target, 'footprint': footprint,
                            'wind': wind, 'fold': fold, **model.best_params_})
    print(target, 'done')
centre_predictions = pd.concat(centre_predictions, ignore_index=True)
centre_predictions.to_csv(CENTRE / 'development_predictions.csv', index=False)
pd.DataFrame(centre_choices).to_csv(CENTRE / 'svr_choices.csv', index=False)


In [ ]:
# 15: paired development errors, not directly comparable with the larger earlier cohort
centre_results = []
for (target, unit, model, wind, footprint), group in centre_predictions.groupby(
        ['target', 'target_unit', 'model', 'wind', 'footprint'], sort=False):
    observed, predicted = group['target_value'].to_numpy(), group['prediction'].to_numpy()
    error = predicted-observed
    centre_results.append({'target': target, 'unit': unit, 'model': model,
        'wind': wind, 'footprint': footprint, 'n': len(group),
        'dates': group['acquisition_date'].nunique(),
        'MAE': np.mean(abs(error)), 'RMSE': np.sqrt(np.mean(error**2)),
        'R2': 1-np.sum(error**2)/np.sum((observed-observed.mean())**2),
        'log10_RMSE': np.sqrt(np.mean((np.log10(predicted)-np.log10(observed))**2))})
centre_results = pd.DataFrame(centre_results)
centre_results.to_csv(CENTRE / 'development_results.csv', index=False)
centre_results.pivot(index=['target', 'unit', 'model', 'wind'],
    columns='footprint', values=['log10_RMSE', 'RMSE', 'R2']).round(3)


The centre pixel does not improve log10-RMSE in any RF/SVR comparison, with or without wind. Without wind, RF SPM RMSE falls slightly from 17.68 to 17.52 mg/L, but Secchi RMSE rises from 1.22 to 1.30 m. SVR physical errors also increase. We use 3 × 3 means in later comparisons.

Both footprints use the same 103 SPM and 58 Secchi observations, one fewer per target than Section 14. Their scores therefore do not directly match the earlier, larger dataset. The extraction record includes the two incomplete centre rows. These are development results. [Run and checks](models/s3_centre_01/run.json).


## 16. Add GLORIA satellite matchups

We test whether the satellite–field pairs from Section 11 help as extra training data. Here RF, SVR and regressions learn directly from satellite reflectance. They do not use the neural weights trained on GLORIA field spectra.

We compare local-only training with local plus GLORIA pairs, using the same local evaluation rows, date groups and eight optical inputs. We exclude wind and use S3 3 × 3 means, so the training data are the only change.

Additions need positive labels, complete inputs, accepted satellite quality and resolved timestamps. They must belong to Section 6's GLORIA training groups, with validation/test groups excluded. This gives 19 S2-SPM observations on five dates, five S2-Secchi observations on one date, and ten S3-SPM observations on four dates. No S3-Secchi additions qualify. Each added observation has equal weight, and its dates do not overlap the local dates.

We add GLORIA rows to training in every evaluation round. SVR's smaller tuning rounds also use them for training, while scoring predictions on local dates. The table compares predictions for the same local measurements, not for the added training observations. We make no reserved-test predictions.


In [ ]:
# 16: distinguish prepared GLORIA data from training-eligible additions
EXPAND = PROJECT / 'models/gloria_expansion_01'
assert not EXPAND.exists(), 'Read the saved run or use a new folder.'
EXPAND.mkdir(parents=True)
gloria_sat = pd.read_csv(PROJECT / 'gloria_satellite/matchups.csv', dtype={'collection': str})
expansion_source_roles = pd.read_csv(PREP / 'gloria_assignments.csv')
gloria_sat = gloria_sat.merge(expansion_source_roles[['GLORIA_ID', 'geographic_group', 'role']].rename(
    columns={'role': 'gloria_role'}), on='GLORIA_ID', how='left', validate='many_to_one')
gloria_sat['input_complete'] = False
for (sensor, target), group in gloria_sat.groupby(['sensor', 'target']):
    columns = ['raw__' + index_columns.get(f, 'Rrs_'+f) for f in selected_inputs[sensor, target]]
    gloria_sat.loc[group.index, 'input_complete'] = np.isfinite(group[columns]).all(axis=1)
gloria_sat['training_eligible'] = (~gloria_sat['time_precision_review']
    & gloria_sat['gloria_role'].eq('train') & gloria_sat['input_complete']
    & gloria_sat['target_value'].gt(0)
    & (gloria_sat['sensor'].eq('S2') | gloria_sat['collection'].eq('003')))
gloria_sat[['row_id', 'GLORIA_ID', 'sensor', 'target', 'geographic_group', 'gloria_role',
    'time_precision_review', 'input_complete', 'training_eligible']].to_csv(
    EXPAND / 'eligibility.csv', index=False)
gloria_added = gloria_sat.loc[gloria_sat['training_eligible']].copy()
gloria_added['target_unit'] = gloria_added['unit']
expansion_local_roles = pd.read_csv(PREP / 'local_assignments.csv')
assert set(gloria_added['acquisition_date']).isdisjoint(expansion_local_roles['acquisition_date'])
expansion_folds = pd.read_csv(PROJECT / 'models/local_baselines_01/fold_assignments.csv')
expansion_ids = expansion_folds.loc[expansion_folds['input_complete'], 'row_id']
expansion_dev = matchups.set_index('row_id').loc[expansion_ids].reset_index()
expansion_dev = expansion_dev.merge(expansion_folds[['row_id', 'fold']], on='row_id', validate='one_to_one')
expansion_counts = expansion_dev.groupby(['sensor', 'target']).agg(
    local_development=('row_id', 'size'), local_dates=('acquisition_date', 'nunique'))
added_counts = gloria_added.groupby(['sensor', 'target']).agg(
    GLORIA_added=('row_id', 'size'), GLORIA_dates=('acquisition_date', 'nunique'))
expansion_counts = expansion_counts.join(added_counts).fillna(0).astype(int)
expansion_counts.to_csv(EXPAND / 'counts.csv')
gloria_added.to_csv(EXPAND / 'training_additions.csv', index=False)
expansion_counts


In [ ]:
# 16: validate on local dates only, with external rows added to training
old_predictions = pd.read_csv(PROJECT / 'models/local_baselines_01/development_predictions.csv')
expansion_predictions = [old_predictions.assign(training='Local only')]
expansion_choices, expansion_power, expansion_inner_rows = [], [], []
for (sensor, target), group in expansion_dev.groupby(['sensor', 'target']):
    extra = gloria_added.loc[gloria_added['sensor'].eq(sensor) & gloria_added['target'].eq(target)]
    if extra.empty:
        print(sensor, target, 'no eligible GLORIA additions')
        continue
    columns = ['raw__' + index_columns.get(f, 'Rrs_'+f) for f in selected_inputs[sensor, target]]
    edge = 'raw__Rrs_' + ('B5' if sensor == 'S2' else 'Oa11')
    for fold in range(3):
        local_train = group.loc[group['fold'].ne(fold)]
        hold = group.loc[group['fold'].eq(fold)]
        train = pd.concat([local_train, extra], ignore_index=True)
        assert set(train['acquisition_date']).isdisjoint(hold['acquisition_date'])
        auxiliary_indices = np.arange(len(local_train), len(train))
        inner_folds = []
        for inner, (a, b) in enumerate(GroupKFold(3, shuffle=True, random_state=42).split(
                local_train, groups=local_train['acquisition_date'])):
            inner_folds.append((np.r_[a, auxiliary_indices], b))
            expansion_inner_rows.append(local_train.iloc[b][['row_id', 'acquisition_date']].assign(
                sensor=sensor, target=target, outer_fold=fold, inner_validation_fold=inner))
        models = {
            'Constant': DummyRegressor(strategy='mean'),
            'Power law': make_pipeline(FunctionTransformer(np.log10), LinearRegression()),
            'Random forest': RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                                                   random_state=42, n_jobs=1),
            'SVR': GridSearchCV(make_pipeline(RobustScaler(), SVR(kernel='rbf', gamma='scale')),
                {'svr__C': [1, 10, 100], 'svr__epsilon': [0.05, 0.1]}, cv=inner_folds,
                scoring='neg_mean_squared_error', n_jobs=1, error_score='raise')}
        for name, model in models.items():
            inputs = [edge] if name == 'Power law' else columns
            model.fit(train[inputs].to_numpy(), np.log10(train['target_value']))
            prediction = 10.0**model.predict(hold[inputs].to_numpy())
            assert np.isfinite(prediction).all() and (prediction > 0).all()
            expansion_predictions.append(hold[['row_id', 'sensor', 'target', 'target_unit',
                'acquisition_date', 'fold', 'target_value']].assign(
                model=name, training='Local + GLORIA', prediction=prediction))
            if name == 'SVR':
                expansion_choices.append({'sensor': sensor, 'target': target, 'fold': fold, **model.best_params_})
            if name == 'Power law':
                linear = model.named_steps['linearregression']
                expansion_power.append({'sensor': sensor, 'target': target, 'fold': fold,
                    'a': 10.0**linear.intercept_, 'b': linear.coef_[0]})
    print(sensor, target, 'done')
expansion_predictions = pd.concat(expansion_predictions, ignore_index=True)
expansion_predictions.to_csv(EXPAND / 'development_predictions.csv', index=False)
pd.DataFrame(expansion_choices).to_csv(EXPAND / 'svr_choices.csv', index=False)
pd.DataFrame(expansion_power).to_csv(EXPAND / 'power_coefficients.csv', index=False)
pd.concat(expansion_inner_rows, ignore_index=True).to_csv(EXPAND / 'local_inner_validation.csv', index=False)


In [ ]:
# 16: paired local validation errors, not scores on the added GLORIA rows
expansion_results = []
for (sensor, target, unit, model, training), group in expansion_predictions.groupby(
        ['sensor', 'target', 'target_unit', 'model', 'training'], sort=False):
    observed, predicted = group['target_value'].to_numpy(), group['prediction'].to_numpy()
    error = predicted-observed
    expansion_results.append({'sensor': sensor, 'target': target, 'unit': unit,
        'model': model, 'training': training, 'n_local_validation': len(group),
        'dates': group['acquisition_date'].nunique(), 'MAE': np.mean(abs(error)),
        'RMSE': np.sqrt(np.mean(error**2)),
        'R2': 1-np.sum(error**2)/np.sum((observed-observed.mean())**2),
        'log10_RMSE': np.sqrt(np.mean((np.log10(predicted)-np.log10(observed))**2))})
expansion_results = pd.DataFrame(expansion_results)
expansion_results.to_csv(EXPAND / 'development_results.csv', index=False)
expansion_results.pivot(index=['sensor', 'target', 'unit', 'model'],
    columns='training', values=['log10_RMSE', 'RMSE', 'R2']).round(3)


The additions do not improve results consistently. RF S2-SPM RMSE rises from 32.76 to 34.77 mg/L and S3-SPM from 17.53 to 17.86 mg/L. S2-Secchi RF RMSE stays around 1.33 m, while log10-RMSE worsens from 0.198 to 0.209.

For the red-edge power law, RMSE changes from 33.48 to 33.47 mg/L for S2-SPM, 1.368 to 1.342 m for S2-Secchi, and 20.38 to 20.64 mg/L for S3-SPM. Log-error falls slightly in all three. These small, mixed changes describe this regression only.

S2-Secchi SVR log10-RMSE improves from 0.230 to 0.211, still worse than local-only RF at 0.198. Its five extra samples cover one date and just 0.9–1.0 m Secchi depth. There is no expanded S3-Secchi comparison because no additions qualified.

We use local-only training as the reference. Each comparison evaluates the same local rows, without new reserved-test predictions. This tests added satellite matchups, separately from the earlier neural pretraining. [Run and checks](models/gloria_expansion_01/run.json).


## 17. Simple statistical fits

We compare other simple optical relationships with the red-edge power law, using the same 270 complete local development observations and saved date groups. Wind and GLORIA additions are excluded. The targets are field SPM or Secchi, with indices used as inputs.

Let $G$, $R$ and $E$ be green, red and red-edge reflectance, and $y$ the target. The candidate families are:

| Family | Equation | Inputs |
| --- | --- | --- |
| Power law | $y=A x^b$, fitted as $\log_{10}y=a+b\log_{10}x$ | $x=E$, $R/G$, $E/R$ or $E/G$, also $R$ for Secchi |
| Exponential, labelled “Log-linear” | $\log_{10}y=a+bx$ | $E/G$ for SPM or $G/E$ for Secchi, NDTI, or ND_rededge_red |
| Three-band regression | $\log_{10}y=a+bG+cR+dE$ | The three unlogged bands |
| Raw linear SPM benchmark | $y=a+bR$ | Red reflectance |

A power law logs both target and input. The exponential model logs the target alone, allowing a signed index such as NDTI as an input. Least squares estimates coefficients by minimising squared differences on the chosen target scale. We convert predictions back without a log-bias correction.

[Alikas & Kratzer (2017), Eq. 1 and Table 2](https://doi.org/10.1016/j.ecolind.2017.02.007) fitted ratio power laws for Secchi, including 560/709 nm. Reversing a ratio changes the exponent's sign within the same family. [Stock (2015), §2.4](https://doi.org/10.1016/j.jag.2015.04.002) recalibrated a log-ratio Secchi relationship regionally. We use these studies to guide local ratio-power fitting, with our own SPM bands and coefficients.

[Knaeps et al. (2010), §5–6](https://doi.org/10.3390/rs2092040) tested exponential band-ratio SPM relationships at different wavelengths. The red-band linear SPM benchmark draws on [Miller & McKee (2004)](https://doi.org/10.1016/j.rse.2004.07.012). The red-band Secchi power law draws on lake results from [Zhang et al. (2021)](https://doi.org/10.1016/j.watres.2021.116844). We added a four-coefficient three-band model to compare a small multivariable fit, following the regional modelling idea in Stock without reproducing its generalised additive model. [Source details](data/statistical_methods_literature.md).

We choose formulas within training dates. Each round excludes one of three date groups for evaluation. The other observations are split into three smaller groups to choose the formula with the lowest pooled squared log-error. We fit the chosen formula to those observations and predict the initially excluded dates. Repeating this gives every development observation a prediction from a model that excluded its date. [Scikit-learn nested validation](https://scikit-learn.org/1.7/auto_examples/model_selection/plot_nested_cross_validation_iris.html).

“Selected power law” pools predictions from the formula chosen in each round, which can differ between rounds. “Selected regression” allows the broader log-target list. These rows do not describe additional models fitted to all observations. The raw linear benchmark is excluded from log-error selection because it can predict zero or negative values. We report those predictions without clipping.

The earlier correlations included reserved observations, so their highest coefficient does not choose the input. We make no new test predictions here. Repeated use of these development groups during exploration limits the independence of the comparisons.


In [ ]:
# 17: reuse local rows and saved date folds
STAT = PROJECT / 'models/statistical_fits_01'
assert not STAT.exists(), 'Read the saved run or use a new folder.'
STAT.mkdir(parents=True)
stat_folds = pd.read_csv(PROJECT / 'models/local_baselines_01/fold_assignments.csv')
stat_data = matchups.merge(stat_folds.loc[stat_folds['input_complete']],
    on=['row_id', 'sensor', 'target', 'acquisition_date'], validate='one_to_one')
stat_inner = pd.read_csv(PROJECT / 'models/local_baselines_01/inner_assignments.csv')
stat_roles = pd.read_csv(PREP / 'local_assignments.csv')
reserved = stat_roles.loc[stat_roles['role'].eq('test')]
assert set(stat_data['row_id']).isdisjoint(reserved['row_id'])
assert set(stat_data['acquisition_date']).isdisjoint(reserved['acquisition_date'])
stat_counts = stat_data.groupby(['sensor', 'target']).agg(
    n=('row_id', 'size'), dates=('acquisition_date', 'nunique'))
stat_counts.to_csv(STAT / 'counts.csv')
stat_counts


In [ ]:
# 17: select regressions only inside each outer training set
stat_records, stat_choices, stat_scores, stat_coefficients = [], [], [], []
for (sensor, target), group in stat_data.groupby(['sensor', 'target']):
    group = group.set_index('row_id')
    bands = ['raw__Rrs_' + b for b in (['B3', 'B4', 'B5'] if sensor == 'S2' else ['Oa06', 'Oa08', 'Oa11'])]
    green, red, edge = bands
    powers = {'Power: red edge': edge, 'Power: red/green': 'raw__red_green_ratio',
        'Power: red edge/red': 'raw__rededge_red_ratio', 'Power: red edge/green': 'raw__rededge_green_ratio'}
    if target == 'Secchi':
        powers['Power: red'] = red
    assert group[list(powers.values())].gt(0).all().all()
    inputs = {name: np.log10(group[[column]]) for name, column in powers.items()}
    ratio = group[['raw__rededge_green_ratio']]
    inputs.update({'Log-linear: ratio': ratio if target == 'SPM' else 1.0/ratio,
        'Log-linear: NDTI': group[['raw__NDTI']],
        'Log-linear: ND red edge/red': group[['raw__ND_rededge_red']],
        'Three bands': group[bands]})
    assert all(np.isfinite(x).all().all() for x in inputs.values())
    log_y = np.log10(group['target_value'])
    for fold in range(3):
        train, hold = group.loc[group['fold'].ne(fold)], group.loc[group['fold'].eq(fold)]
        inner = stat_inner.loc[stat_inner['sensor'].eq(sensor)
            & stat_inner['target'].eq(target) & stat_inner['outer_fold'].eq(fold)].set_index('row_id')
        assert set(inner.index) == set(train.index)
        scores, predictions = {}, {}
        for name, x in inputs.items():
            squared_errors = []
            for inner_fold in range(3):
                a = inner.index[inner['inner_validation_fold'].ne(inner_fold)]
                b = inner.index[inner['inner_validation_fold'].eq(inner_fold)]
                assert set(group.loc[a, 'acquisition_date']).isdisjoint(group.loc[b, 'acquisition_date'])
                model = LinearRegression().fit(x.loc[a], log_y.loc[a])
                squared_errors.extend((model.predict(x.loc[b])-log_y.loc[b])**2)
            scores[name] = np.mean(squared_errors)
            stat_scores.append([sensor, target, fold, name, np.sqrt(scores[name])])
            model = LinearRegression().fit(x.loc[train.index], log_y.loc[train.index])
            prediction = 10.0**model.predict(x.loc[hold.index])
            assert np.isfinite(prediction).all() and (prediction > 0).all()
            outside = ((x.loc[hold.index] < x.loc[train.index].min()) |
                       (x.loc[hold.index] > x.loc[train.index].max())).any(axis=1)
            part = hold[['sensor', 'target', 'target_unit', 'acquisition_date', 'fold', 'target_value']].assign(
                model=name, prediction=prediction, outside_training_range=outside)
            predictions[name] = part.reset_index()
            stat_records.append(predictions[name])
            stat_coefficients.append({'sensor': sensor, 'target': target, 'fold': fold, 'model': name,
                'intercept': model.intercept_, 'response': 'log10',
                **{column: value for column, value in zip(x.columns, model.coef_)}})
        for label, candidates in [('Selected power law', list(powers)), ('Selected regression', list(inputs))]:
            chosen = min(candidates, key=scores.get)
            stat_choices.append([sensor, target, fold, label, chosen, np.sqrt(scores[chosen])])
            stat_records.append(predictions[chosen].assign(model=label))
        if target == 'SPM':
            model = LinearRegression().fit(train[[red]], train['target_value'])
            stat_records.append(hold[['sensor', 'target', 'target_unit', 'acquisition_date', 'fold',
                'target_value']].assign(model='Linear: red', prediction=model.predict(hold[[red]])).reset_index())
            stat_coefficients.append({'sensor': sensor, 'target': target, 'fold': fold, 'model': 'Linear: red',
                'intercept': model.intercept_, 'response': 'physical', red: model.coef_[0]})
stat_predictions = pd.concat(stat_records, ignore_index=True)
stat_predictions.to_csv(STAT / 'development_predictions.csv', index=False)
pd.DataFrame(stat_choices, columns=['sensor', 'target', 'fold', 'selection', 'model', 'inner_log10_RMSE']).to_csv(
    STAT / 'model_choices.csv', index=False)
pd.DataFrame(stat_scores, columns=['sensor', 'target', 'fold', 'model', 'inner_log10_RMSE']).to_csv(
    STAT / 'inner_scores.csv', index=False)
pd.DataFrame(stat_coefficients).to_csv(STAT / 'coefficients.csv', index=False)


In [ ]:
# 17: outer-fold errors, with the previous RF and constant as references
stat_reference = pd.read_csv(PROJECT / 'models/local_baselines_01/development_predictions.csv')
stat_comparison = pd.concat([stat_predictions,
    stat_reference.loc[stat_reference['model'].isin(['Random forest', 'Constant'])]], ignore_index=True)
stat_results = []
for (sensor, target, unit, model), group in stat_comparison.groupby(
        ['sensor', 'target', 'target_unit', 'model'], sort=False):
    y, prediction = group['target_value'].to_numpy(), group['prediction'].to_numpy()
    error = prediction-y
    stat_results.append({'sensor': sensor, 'target': target, 'unit': unit, 'model': model,
        'n': len(group), 'dates': group['acquisition_date'].nunique(),
        'MAE': np.mean(abs(error)), 'RMSE': np.sqrt(np.mean(error**2)), 'bias': np.mean(error),
        'R2': 1-np.sum(error**2)/np.sum((y-y.mean())**2),
        'nonpositive_predictions': int((prediction <= 0).sum()),
        'log10_RMSE': np.sqrt(np.mean((np.log10(prediction)-np.log10(y))**2)) if (prediction > 0).all() else np.nan})
stat_results = pd.DataFrame(stat_results)
stat_results.to_csv(STAT / 'development_results.csv', index=False)
stat_results.loc[stat_results['model'].isin([
    'Selected power law', 'Selected regression', 'Power: red edge', 'Random forest'])].set_index(
    ['sensor', 'target', 'unit', 'model'])[['n', 'dates', 'log10_RMSE', 'RMSE', 'R2']].round(3)


Red/green is selected in all three SPM rounds. Relative to the red-edge power law, S3 log10-RMSE falls from 0.463 to 0.366 and physical RMSE from 20.38 to 15.93 mg/L, compared with RF at 17.53 mg/L. For S2, log-error falls from 0.440 to 0.409, but RMSE rises from 33.48 to 34.27 mg/L. Lower proportional error does not always mean lower absolute error.

S2-Secchi selection chooses red edge. For S3-Secchi, power-law selection reduces RMSE from 5.39 to 1.77 m, but RF is lower at 1.33 m. The broader regression list offers no consistent gain.

Some models extrapolate badly, especially the exponential green/red-edge Secchi model. We include their extreme predictions in the results. All comparisons use the same 270 development observations and saved groups, without further row loss or test predictions. We have not established statistical significance for the score differences. [Errors](models/statistical_fits_01/development_results.csv), [training-only formula choices](models/statistical_fits_01/model_choices.csv), [checks](models/statistical_fits_01/run.json).

We chose separate S2/S3 red/green power laws for the final SPM application. Secchi results were useful, particularly with RF, but were outside the final mapping scope. The [final-method notebook](Coastal_final_method.ipynb) covers calibration, reserved-date evaluation and maps. Counts here describe each experiment as run, even when a later model can use more rows because it needs fewer inputs.
